# SIGNOVA KArSL: Colab Experiments and Verification

Research prototype for 30 isolated KArSL sign classes. The selected model is a previously trained bidirectional GRU, loaded from the private reproducibility bundle and verified in Colab.

| Saved verification result | Value |
|---|---:|
| Final package tests passed | 17 |
| Final package tests skipped | 0 |
| GRU validation samples | 450 |
| GRU validation accuracy | 32.00% |
| GRU validation Macro F1 | 0.2755 |
| GRU validation Top-5 accuracy | 62.44% |

The real validation Pose sample passed preprocessing parity and GRU inference. Independent one-epoch smoke training passed without replacing the selected checkpoint. Smoke-training scores describe a separate small test run.

## Execution record

Sections 1–5 retain historical exploration and model code. Many of those cells have no saved outputs in this export; they are not presented as newly executed experiments. The historical missing `pose_format` error is retained, and the later real-Pose and final tests show successful execution after dependency setup.

Sections 6–14 contain the saved Colab continuation results. Code and ML outputs were not rerun during editorial cleanup. Unrelated Drive directory listings and the transient upload widget were omitted; saved ML results are unchanged.

Use `README.md`, `COLAB_RUNBOOK.md`, and `06_KArSL_ML_Experiments.ipynb` for package reproduction. This notebook records a stateful workflow, including one-time extraction and historical training/test cells; it is not intended for an unattended Run all. The import cell expects the original private ZIP and its original checksum. The last archive log predates adding this notebook to the delivery archives.

Accuracy remains limited. This is not a 502-class model or a production translator. The prediction contract still needs backend agreement; OOV evaluation and source/release review remain pending.


## 1. Historical data and source inspection


In [11]:
from google.colab import drive
from pathlib import Path
import zipfile
import pandas as pd

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive")
package = root / "SIGNOVA/KArSL/SIGNOVA_KArSL_Cleaned_Full.zip"

assert package.is_file(), "KArSL package not found"

with zipfile.ZipFile(package) as z:
    with z.open("karsl/data/pose/karsl_pose_manifest.csv") as f:
        pose_manifest = pd.read_csv(f)

    with z.open("karsl/data/pose/karsl_pose_signer_folds.csv") as f:
        pose_folds = pd.read_csv(f)

fold_1 = pose_folds[pose_folds["fold"] == "fold_1"].copy()

print("=== KARSL POSE DATA ===")
print("Pose samples:", len(pose_manifest))
print("Classes:", pose_manifest["sign_id"].nunique())
print("Signers:", pose_manifest["signer_id"].nunique())

print("\n=== FOLD 1 ===")
print(fold_1["split"].value_counts().to_string())

print("\n=== EXAMPLE FILE ===")
print(pose_manifest["pose_file"].iloc[0])

print("\nStatus: METADATA READY")
print("Files modified: NO")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== KARSL POSE DATA ===
Pose samples: 75515
Classes: 502
Signers: 3

=== FOLD 1 ===
split
validation    25316
test          25112
train         25087

=== EXAMPLE FILE ===
pose_dir/sign_0001_s1_sess1_001.pose

Status: METADATA READY
Files modified: NO


In [12]:
from pathlib import Path
from itertools import islice

folders = [
    Path("/content/drive/MyDrive/SIGNOVA/KArSL/raw"),
    Path("/content/drive/MyDrive/SIGNOVA/KArSL/keypoints"),
    Path("/content/drive/MyDrive/SIGNOVA/KArSL/handoff"),
    Path("/content/drive/MyDrive/SIGNOVA/KArSL"),
]

print("=== POSE FILE LOCATION CHECK ===")

for folder in folders:
    print(f"\n{folder}")

    if not folder.is_dir():
        print("NOT FOUND")
        continue

    items = list(islice(folder.iterdir(), 60))

    for item in items:
        if item.is_dir():
            print("[DIR]", item.name)
            children = list(islice(item.iterdir(), 5))
            for child in children:
                print("   ", "[DIR]" if child.is_dir() else "[FILE]", child.name)
        else:
            print("[FILE]", item.name)

print("\nOriginal data modified: NO")
print("GitHub modified: NO")


=== POSE FILE LOCATION CHECK ===

/content/drive/MyDrive/SIGNOVA/KArSL/raw

/content/drive/MyDrive/SIGNOVA/KArSL/keypoints
[DIR] pilot
    [FILE] KARSL_001369.npy
    [FILE] KARSL_001370.npy
    [FILE] KARSL_001371.npy
    [FILE] KARSL_001372.npy
    [FILE] KARSL_001373.npy
[DIR] kaggle
    [DIR] karsl-502
    [DIR] .complete

/content/drive/MyDrive/SIGNOVA/KArSL/handoff
[FILE] dataset_sources.csv
[FILE] download_manifest.csv
[FILE] metadata_arsl.csv
[FILE] class_mapping_v2.csv
[FILE] qc_flags.csv
[FILE] split_manifest_v2.csv
[FILE] label_issues.md
[FILE] count_anomalies.csv
[FILE] duplicate_report.csv
[FILE] dataset_tests_report.csv
[FILE] final_readiness_checks.csv
[FILE] open_review_cases.csv
[FILE] split_manifest_pass_only.csv
[FILE] split_manifest_3way.csv
[FILE] split_distribution.csv
[FILE] split_class_gaps.csv
[FILE] split_excluded_classes.csv
[FILE] class_mapping_v3.csv
[FILE] benchmark_v3_summary.md
[FILE] dataset_tests_v3.csv
[FILE] dataset_card.md
[FILE] handoff.md
[FILE] q

In [13]:
from pathlib import Path
from collections import deque

root = Path("/content/drive/MyDrive/SIGNOVA/KArSL/keypoints/kaggle")

queue = deque([(root, 0)])
visited = 0
extensions = set()

print("=== EXISTING KAGGLE DATA ===")

while queue and visited < 20:
    folder, depth = queue.popleft()

    if not folder.is_dir():
        continue

    visited += 1
    items = sorted(folder.iterdir(), key=lambda p: p.name)

    print(f"\n{folder.relative_to(root)}")
    print("Items:", len(items))

    for item in items[:12]:
        print(" ", "[DIR]" if item.is_dir() else "[FILE]", item.name)

    for item in items:
        if item.is_file():
            extensions.add(item.suffix.lower())

    if depth < 3:
        for item in items:
            if item.is_dir() and len(queue) < 30:
                queue.append((item, depth + 1))

print("\n=== FILE TYPES FOUND ===")
print(sorted(extensions))
print("Folders inspected:", visited)
print("Original data modified: NO")
print("GitHub modified: NO")


=== EXISTING KAGGLE DATA ===

.
Items: 2
  [DIR] .complete
  [DIR] karsl-502

.complete
Items: 1
  [DIR] datasets

karsl-502
Items: 3
  [DIR] 01
  [DIR] 02
  [DIR] 03

.complete/datasets
Items: 1
  [DIR] yousefdotpy

karsl-502/01
Items: 2
  [DIR] test
  [DIR] train

karsl-502/02
Items: 2
  [DIR] test
  [DIR] train

karsl-502/03
Items: 1
  [DIR] train

.complete/datasets/yousefdotpy
Items: 1
  [DIR] blablabla

karsl-502/01/test
Items: 112
  [DIR] 0289
  [DIR] 0290
  [DIR] 0291
  [DIR] 0292
  [DIR] 0293
  [DIR] 0294
  [DIR] 0295
  [DIR] 0296
  [DIR] 0297
  [DIR] 0298
  [DIR] 0299
  [DIR] 0300

karsl-502/01/train
Items: 112
  [DIR] 0289
  [DIR] 0290
  [DIR] 0291
  [DIR] 0292
  [DIR] 0293
  [DIR] 0294
  [DIR] 0295
  [DIR] 0296
  [DIR] 0297
  [DIR] 0298
  [DIR] 0299
  [DIR] 0300

karsl-502/02/test
Items: 112
  [DIR] 0289
  [DIR] 0290
  [DIR] 0291
  [DIR] 0292
  [DIR] 0293
  [DIR] 0294
  [DIR] 0295
  [DIR] 0296
  [DIR] 0297
  [DIR] 0298
  [DIR] 0299
  [DIR] 0300

karsl-502/02/train
Items: 11

In [14]:
from pathlib import Path
from itertools import islice

root = Path("/content/drive/MyDrive/SIGNOVA/KArSL/keypoints/kaggle")

paths = [
    root / "karsl-502/01/train/0289",
    root / "karsl-502/02/test/0289",
    root / ".complete/datasets/yousefdotpy/blablabla"
]

for folder in paths:
    print("\nFOLDER:", folder)

    if not folder.is_dir():
        print("NOT FOUND")
        continue

    items = list(islice(folder.iterdir(), 10))

    for item in items:
        print(" ", item.name, "DIR" if item.is_dir() else "FILE")

        if item.is_dir():
            for child in islice(item.iterdir(), 5):
                print("    ", child.name, child.suffix)

        elif item.is_file():
            print("    Extension:", item.suffix)

print("\nOriginal data modified: NO")



FOLDER: /content/drive/MyDrive/SIGNOVA/KArSL/keypoints/kaggle/karsl-502/01/train/0289
  lh_keypoints DIR
     03_01_0289_(15_08_18_13_37_49)_c.npy .npy
     03_01_0289_(15_08_18_13_37_55)_c.npy .npy
     03_01_0289_(15_08_18_13_37_57)_c.npy .npy
     03_01_0289_(15_08_18_13_37_59)_c.npy .npy
     03_01_0289_(15_08_18_13_38_02)_c.npy .npy
  pose_keypoints DIR
     03_01_0289_(15_08_18_13_37_49)_c.npy .npy
     03_01_0289_(15_08_18_13_37_55)_c.npy .npy
     03_01_0289_(15_08_18_13_37_57)_c.npy .npy
     03_01_0289_(15_08_18_13_37_59)_c.npy .npy
     03_01_0289_(15_08_18_13_38_02)_c.npy .npy
  rh_keypoints DIR
     03_01_0289_(15_08_18_13_37_49)_c.npy .npy
     03_01_0289_(15_08_18_13_37_55)_c.npy .npy
     03_01_0289_(15_08_18_13_37_57)_c.npy .npy
     03_01_0289_(15_08_18_13_37_59)_c.npy .npy
     03_01_0289_(15_08_18_13_38_02)_c.npy .npy

FOLDER: /content/drive/MyDrive/SIGNOVA/KArSL/keypoints/kaggle/karsl-502/02/test/0289
  lh_keypoints DIR
     03_02_0289_(14_11_17_20_57_50)_c.npy .n

In [15]:
import requests

dataset = "tfmohamedyahia/kars502-pose-75"
url = f"https://www.kaggle.com/api/v1/datasets/view/{dataset}"

response = requests.get(url, timeout=30)

print("=== KAGGLE DATASET CHECK ===")
print("HTTP status:", response.status_code)

if response.ok:
    info = response.json()
    files = info.get("resources", [])

    print("Dataset:", info.get("title"))
    print("Version:", info.get("currentVersionNumber"))
    print("Files listed:", len(files))

    for item in files[:15]:
        print(" -", item.get("path", item.get("name")))
else:
    print("Dataset metadata is not accessible")
    print("Response:", response.text[:300])

print("\nFull dataset downloaded: NO")
print("Google Drive modified: NO")


=== KAGGLE DATASET CHECK ===
HTTP status: 200
Dataset: karsl502-pose-video-75k
Version: 1
Files listed: 0

Full dataset downloaded: NO
Google Drive modified: NO


In [16]:
import requests

url = "https://www.kaggle.com/api/v1/datasets/list/tfmohamedyahia/kars502-pose-75"

response = requests.get(url, params={"pageSize": 10}, timeout=30)

print("=== KAGGLE FILE LIST ===")
print("HTTP status:", response.status_code)

if response.ok:
    data = response.json()

    if isinstance(data, dict):
        print("Response keys:", list(data.keys()))
        files = data.get("files", data.get("datasetFiles", []))
    elif isinstance(data, list):
        files = data
    else:
        files = []

    print("Files returned:", len(files))

    for item in files[:10]:
        if isinstance(item, dict):
            print(item.get("name", item.get("path", str(item)[:200])))
        else:
            print(str(item)[:200])

    if not files:
        print("Response preview:", str(data)[:700])
else:
    print("Response:", response.text[:500])

print("Dataset downloaded: NO")


=== KAGGLE FILE LIST ===
HTTP status: 200
Response keys: ['nextPageTokenNullable', 'datasetFiles', 'errorMessage', 'hasErrorMessage', 'nextPageToken', 'hasNextPageToken']
Files returned: 10
content/kars502-pose-75k/labels_mapping.json
content/kars502-pose-75k/metadata.json
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_001.pose
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_002.pose
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_003.pose
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_004.pose
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_005.pose
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_006.pose
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_007.pose
content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_008.pose
Dataset downloaded: NO


### Historical dependency failure

The saved error below occurred before `pose-format` was installed. Later real-Pose checks and the final 17-test run passed.


In [17]:
from pathlib import Path
import kagglehub
from pose_format import Pose

dataset = "tfmohamedyahia/kars502-pose-75"

filename = "content/kars502-pose-75k/pose_dir/sign_0001_s1_sess1_001.pose"

file_path = Path(
    kagglehub.dataset_download(dataset, path=filename)
)

assert file_path.is_file()

with file_path.open("rb") as f:
    pose = Pose.read(f.read())

print("=== POSE SAMPLE TEST ===")
print("File:", file_path.name)
print("Size KB:", round(file_path.stat().st_size / 1024, 2))
print("Data shape:", pose.body.data.shape)
print("Confidence shape:", pose.body.confidence.shape)
print("Status: PASS")
print("Full dataset downloaded: NO")
print("Google Drive modified: NO")


ModuleNotFoundError: No module named 'pose_format'

## 2. Historical Pose inspection and preprocessing


In [ ]:
import numpy as np

print("=== POSE STRUCTURE ===")
print("Data shape:", pose.body.data.shape)
print("Confidence shape:", pose.body.confidence.shape)

print("\n=== LANDMARK COMPONENTS ===")
offset = 0

for component in pose.header.components:
    count = len(component.points)
    print(component.name, "| Points:", count, "| Indices:", offset, "to", offset + count - 1)
    offset += count

print("\n=== DATA QUALITY ===")
data = np.asarray(pose.body.data)
confidence = np.asarray(pose.body.confidence)

print("Finite coordinates:", round(np.isfinite(data).mean() * 100, 2), "%")
print("Positive confidence:", round((confidence > 0).mean() * 100, 2), "%")
print("Coordinate dtype:", data.dtype)

print("\n=== TRAINING MANIFEST ===")
print("Pose manifest columns:", pose_manifest.columns.tolist())
print("Fold columns:", pose_folds.columns.tolist())

print("\nSample manifest rows:")
print(pose_manifest.head(2).to_string(index=False))

print("\nSample fold rows:")
print(pose_folds.head(2).to_string(index=False))

print("\nStatus: READY")
print("New downloads: NO")
print("Google Drive modified: NO")


In [ ]:
import numpy as np

data = np.asarray(pose.body.data)[:, 0]
conf = np.asarray(pose.body.confidence)[:, 0]

components = {
    "Body": slice(0, 33),
    "Face": slice(33, 501),
    "Left Hand": slice(501, 522),
    "Right Hand": slice(522, 543)
}

print("=== COORDINATE RANGES ===")

for name, indices in components.items():
    xyz = data[:, indices, :]
    scores = conf[:, indices]
    valid = (scores > 0.5) & np.isfinite(xyz).all(axis=-1)

    print(f"\n{name}")
    print("Valid points:", int(valid.sum()))

    if valid.any():
        points = xyz[valid]
        print("X:", round(float(points[:, 0].min()), 3), "to", round(float(points[:, 0].max()), 3))
        print("Y:", round(float(points[:, 1].min()), 3), "to", round(float(points[:, 1].max()), 3))
        print("Z:", round(float(points[:, 2].min()), 3), "to", round(float(points[:, 2].max()), 3))

frame = len(data) // 2

print("\n=== SAMPLE FRAME ===")
print("Frame:", frame)
print("Left shoulder:", data[frame, 11].round(3))
print("Right shoulder:", data[frame, 12].round(3))
print("Left wrist:", data[frame, 15].round(3))
print("Right wrist:", data[frame, 16].round(3))
print("Left hand wrist:", data[frame, 501].round(3))
print("Right hand wrist:", data[frame, 522].round(3))

print("\nStatus: PASS")
print("Downloads: NO")
print("Files modified: NO")


In [ ]:
import numpy as np

BODY_INDICES = [11, 12, 13, 14, 15, 16, 23, 24]
HAND_INDICES = list(range(501, 543))
LANDMARK_INDICES = BODY_INDICES + HAND_INDICES

TARGET_FRAMES = 32
CONFIDENCE_THRESHOLD = 0.5
FEATURE_DIM = len(LANDMARK_INDICES) * 3

def preprocess_pose(pose_object):
    data = np.asarray(pose_object.body.data, dtype=np.float32)
    confidence = np.asarray(pose_object.body.confidence, dtype=np.float32)

    if data.ndim != 4 or data.shape[1:] != (1, 543, 3):
        raise ValueError(f"Unexpected pose shape: {data.shape}")

    if confidence.shape != data.shape[:3]:
        raise ValueError(f"Unexpected confidence shape: {confidence.shape}")

    if data.shape[0] == 0:
        raise ValueError("Empty pose sequence")

    coordinates = data[:, 0, :, :2]
    scores = confidence[:, 0, :]

    shoulder_valid = (
        (scores[:, 11] > CONFIDENCE_THRESHOLD)
        & (scores[:, 12] > CONFIDENCE_THRESHOLD)
        & np.isfinite(coordinates[:, 11]).all(axis=1)
        & np.isfinite(coordinates[:, 12]).all(axis=1)
    )

    if not shoulder_valid.any():
        raise ValueError("No valid shoulder reference found")

    left_shoulder = coordinates[shoulder_valid, 11]
    right_shoulder = coordinates[shoulder_valid, 12]

    centers = (left_shoulder + right_shoulder) / 2
    widths = np.linalg.norm(left_shoulder - right_shoulder, axis=1)

    center = np.median(centers, axis=0)
    valid_widths = widths[widths > 1e-6]

    if len(valid_widths) == 0:
        raise ValueError("Invalid shoulder width")

    scale = float(np.median(valid_widths))

    selected = coordinates[:, LANDMARK_INDICES, :]
    selected_scores = scores[:, LANDMARK_INDICES]

    valid = (
        (selected_scores > CONFIDENCE_THRESHOLD)
        & np.isfinite(selected).all(axis=2)
    )

    normalized = (selected - center) / scale
    normalized = np.clip(normalized, -5.0, 5.0)
    normalized = np.where(valid[..., None], normalized, 0.0)

    features = np.concatenate(
        [normalized, valid[..., None].astype(np.float32)],
        axis=2
    )

    frame_indices = np.rint(
        np.linspace(0, len(features) - 1, TARGET_FRAMES)
    ).astype(int)

    features = features[frame_indices]
    features = features.reshape(TARGET_FRAMES, FEATURE_DIM)

    return features.astype(np.float32)


X = preprocess_pose(pose)

print("=== KARSL PREPROCESSING TEST ===")
print("Input:", pose.body.data.shape)
print("Output:", X.shape)
print("Expected:", (32, 150))
print("Data type:", X.dtype)
print("NaN values:", np.isnan(X).sum())
print("Infinite values:", np.isinf(X).sum())

visibility = X.reshape(32, 50, 3)[:, :, 2]

print("\n=== LANDMARK VISIBILITY ===")
print("Body:", round(visibility[:, :8].mean() * 100, 1), "%")
print("Left hand:", round(visibility[:, 8:29].mean() * 100, 1), "%")
print("Right hand:", round(visibility[:, 29:50].mean() * 100, 1), "%")

assert X.shape == (32, 150)
assert np.isfinite(X).all()

print("\nStatus: PASS")
print("Original data modified: NO")
print("GitHub modified: NO")


In [ ]:
import kagglehub
import numpy as np
import pandas as pd
from pathlib import Path
from pose_format import Pose

fold_1 = pose_folds[pose_folds["fold"] == "fold_1"].copy()

counts = fold_1.groupby(
    ["sign_id", "split"]
).size().unstack(fill_value=0)

required_splits = ["train", "validation", "test"]

eligible = counts[
    (counts.reindex(columns=required_splits, fill_value=0) >= 2).all(axis=1)
]

assert len(eligible) >= 3, "Not enough classes for the pilot"

selected_classes = eligible.index[:3].tolist()

pilot_rows = (
    fold_1[fold_1["sign_id"].isin(selected_classes)]
    .groupby(["sign_id", "split"], group_keys=False)
    .sample(n=2, random_state=42)
    .sort_values(["sign_id", "split"])
)

X_pilot = []
y_pilot = []
pilot_records = []

for row in pilot_rows.itertuples(index=False):
    file_path = Path(
        kagglehub.dataset_download(
            "tfmohamedyahia/kars502-pose-75",
            path="content/kars502-pose-75k/" + row.pose_file
        )
    )

    with file_path.open("rb") as f:
        sample_pose = Pose.read(f.read())

    features = preprocess_pose(sample_pose)

    assert features.shape == (32, 150)
    assert np.isfinite(features).all()

    X_pilot.append(features)
    y_pilot.append(row.sign_id)

    pilot_records.append({
        "class": row.sign_id,
        "signer": row.signer_id,
        "split": row.split,
        "frames": sample_pose.body.data.shape[0],
        "hand_left_visibility": float(
            features.reshape(32, 50, 3)[:, 8:29, 2].mean()
        ),
        "hand_right_visibility": float(
            features.reshape(32, 50, 3)[:, 29:50, 2].mean()
        )
    })

X_pilot = np.stack(X_pilot)
y_pilot = np.asarray(y_pilot)
pilot_report = pd.DataFrame(pilot_records)

print("=== KARSL PREPROCESSING PILOT ===")
print("Samples:", len(X_pilot))
print("Classes:", selected_classes)
print("X shape:", X_pilot.shape)
print("Y shape:", y_pilot.shape)
print("Finite values:", np.isfinite(X_pilot).all())

print("\n=== SPLIT DISTRIBUTION ===")
print(pilot_report.groupby(["split", "signer"]).size().to_string())

print("\n=== HAND VISIBILITY ===")
print(
    pilot_report[
        ["hand_left_visibility", "hand_right_visibility"]
    ].mean().round(3).to_string()
)

print("\n=== FRAME COUNTS ===")
print(pilot_report["frames"].describe().round(1).to_string())

print("\nStatus: PASS")
print("Full dataset downloaded: NO")
print("Google Drive modified: NO")
print("GitHub modified: NO")


## 3. Historical 10-class baseline experiments


In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import joblib
from pathlib import Path
from pose_format import Pose
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

fold_1 = pose_folds[pose_folds["fold"] == "fold_1"].copy()

counts = fold_1.groupby(["sign_id", "split"]).size().unstack(fill_value=0)

eligible = counts[
    (counts["train"] >= 12) &
    (counts["validation"] >= 6)
]

classes = eligible.index[:10].tolist()
assert len(classes) == 10

train_rows = (
    fold_1[
        (fold_1["sign_id"].isin(classes)) &
        (fold_1["split"] == "train")
    ]
    .groupby("sign_id", group_keys=False)
    .sample(n=12, random_state=42)
)

val_rows = (
    fold_1[
        (fold_1["sign_id"].isin(classes)) &
        (fold_1["split"] == "validation")
    ]
    .groupby("sign_id", group_keys=False)
    .sample(n=6, random_state=42)
)

assert set(train_rows["pose_file"]).isdisjoint(set(val_rows["pose_file"]))
assert set(train_rows["signer_id"]).isdisjoint(set(val_rows["signer_id"]))

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
output_dir.mkdir(parents=True, exist_ok=True)

cache_path = output_dir / "pose_10_classes_features_v1.npz"

if cache_path.is_file():
    cached = np.load(cache_path)
    X_train = cached["X_train"]
    y_train = cached["y_train"]
    X_val = cached["X_val"]
    y_val = cached["y_val"]
    print("Loaded existing feature cache")
else:
    def load_features(rows, split_name):
        features = []
        labels = []

        for i, row in enumerate(rows.itertuples(index=False), start=1):
            path = Path(
                kagglehub.dataset_download(
                    "tfmohamedyahia/kars502-pose-75",
                    path="content/kars502-pose-75k/" + row.pose_file
                )
            )

            with path.open("rb") as f:
                sample = Pose.read(f.read())

            features.append(preprocess_pose(sample).reshape(-1))
            labels.append(row.sign_id)

            if i % 30 == 0:
                print(f"{split_name}: {i}/{len(rows)}")

        return np.stack(features), np.asarray(labels)

    X_train, y_train = load_features(train_rows, "Train")
    X_val, y_val = load_features(val_rows, "Validation")

    np.savez_compressed(
        cache_path,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val
    )

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42)
)

model.fit(X_train, y_train)
predictions = model.predict(X_val)

accuracy = accuracy_score(y_val, predictions)
macro_f1 = f1_score(y_val, predictions, average="macro", zero_division=0)

model_path = output_dir / "karsl_logistic_regression_10_classes_v1.joblib"
joblib.dump(model, model_path)

print("\n=== FIRST KARSL MODEL ===")
print("Classes:", classes)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Input features:", X_train.shape[1])
print("Validation Accuracy:", round(accuracy * 100, 2), "%")
print("Validation Macro F1:", round(macro_f1, 4))
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_val, predictions, zero_division=0))
print("\nModel saved:", model_path)
print("Feature cache saved:", cache_path)
print("GitHub modified: NO")
print("Status: BASELINE TRAINED")


In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
cache_path = output_dir / "pose_10_classes_features_v1.npz"

with np.load(cache_path) as data:
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_val = data["X_val"]
    y_val = data["y_val"]

models = {
    "Logistic Regression": joblib.load(
        output_dir / "karsl_logistic_regression_10_classes_v1.joblib"
    ),
    "SVM RBF": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=10, gamma="scale", random_state=42)
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

results = []
best_name = None
best_score = -1
best_model = None

for name, classifier in models.items():
    if name != "Logistic Regression":
        classifier.fit(X_train, y_train)

    predictions = classifier.predict(X_val)

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": round(accuracy, 4),
        "Macro F1": round(macro_f1, 4)
    })

    if macro_f1 > best_score:
        best_score = macro_f1
        best_name = name
        best_model = classifier

comparison = pd.DataFrame(results).sort_values(
    "Macro F1", ascending=False
)

comparison.to_csv(
    output_dir / "baseline_model_comparison.csv",
    index=False
)

joblib.dump(
    best_model,
    output_dir / "best_10_classes_baseline.joblib"
)

print("=== KARSL BASELINE COMPARISON ===")
print(comparison.to_string(index=False))

print("\nBest model:", best_name)
print("Best Macro F1:", round(best_score, 4))
print("Classes:", len(np.unique(y_train)))
print("New downloads: NO")
print("Training completed: YES")
print("GitHub modified: NO")


In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import joblib
from pathlib import Path
from pose_format import Pose
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
output_dir.mkdir(parents=True, exist_ok=True)

cache_path = output_dir / "pose_10_classes_features_v2.npz"
classes = list(range(1, 11))

fold_1 = pose_folds[
    (pose_folds["fold"] == "fold_1") &
    (pose_folds["sign_id"].isin(classes))
].copy()

def select_samples(split, count):
    rows = fold_1[fold_1["split"] == split]
    assert (rows.groupby("sign_id").size() >= count).all()
    return rows.groupby("sign_id", group_keys=False).sample(
        n=count, random_state=42
    )

train_rows = select_samples("train", 30)
val_rows = select_samples("validation", 15)

assert set(train_rows["signer_id"]).isdisjoint(set(val_rows["signer_id"]))
assert set(train_rows["pose_file"]).isdisjoint(set(val_rows["pose_file"]))

def load_samples(rows, name):
    features, labels = [], []

    for i, row in enumerate(rows.itertuples(index=False), 1):
        file_path = Path(kagglehub.dataset_download(
            "tfmohamedyahia/kars502-pose-75",
            path="content/kars502-pose-75k/" + row.pose_file
        ))

        with file_path.open("rb") as f:
            sample = Pose.read(f.read())

        features.append(preprocess_pose(sample).reshape(-1))
        labels.append(row.sign_id)

        if i % 30 == 0:
            print(f"{name}: {i}/{len(rows)}")

    return np.stack(features), np.asarray(labels)

if cache_path.is_file():
    with np.load(cache_path) as cached:
        X_train = cached["X_train"]
        y_train = cached["y_train"]
        X_val = cached["X_val"]
        y_val = cached["y_val"]
    print("Loaded saved features")
else:
    X_train, y_train = load_samples(train_rows, "Train")
    X_val, y_val = load_samples(val_rows, "Validation")

    np.savez_compressed(
        cache_path,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val
    )

assert X_train.shape == (300, 4800)
assert X_val.shape == (150, 4800)
assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()

model_v2 = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=10, gamma="scale")
)

model_v2.fit(X_train, y_train)
predictions = model_v2.predict(X_val)

accuracy = accuracy_score(y_val, predictions)
macro_f1 = f1_score(y_val, predictions, average="macro", zero_division=0)

joblib.dump(
    model_v2,
    output_dir / "karsl_svm_10_classes_v2.joblib"
)

print("\n=== KARSL SVM V2 ===")
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Classes:", len(np.unique(y_train)))
print("Validation Accuracy:", round(accuracy * 100, 2), "%")
print("Validation Macro F1:", round(macro_f1, 4))
print("Previous SVM Accuracy: 31.67%")
print("Previous SVM Macro F1: 0.2904")
print("Model saved: YES")
print("Features saved: YES")
print("GitHub modified: NO")


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import copy
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
cache_path = output_dir / "pose_10_classes_features_v2.npz"

with np.load(cache_path) as data:
    X_train = data["X_train"].reshape(-1, 32, 150)
    y_train = data["y_train"]
    X_val = data["X_val"].reshape(-1, 32, 150)
    y_val = data["y_val"]

classes = sorted(np.unique(y_train).tolist())
label_to_index = {label: i for i, label in enumerate(classes)}

y_train_encoded = np.array([label_to_index[y] for y in y_train])
y_val_encoded = np.array([label_to_index[y] for y in y_val])

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32).transpose(1, 2),
    torch.tensor(y_train_encoded, dtype=torch.long)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32).transpose(1, 2),
    torch.tensor(y_val_encoded, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

class SignCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(150, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = SignCNN(len(classes)).to(device)

optimizer = torch.optim.AdamW(cnn.parameters(), lr=0.001, weight_decay=0.001)
criterion = nn.CrossEntropyLoss()

best_f1 = -1
best_state = None
best_epoch = 0
patience = 7
wait = 0

print("=== KARSL 1D CNN TRAINING ===")
print("Device:", device)
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Classes:", classes)

for epoch in range(1, 41):
    cnn.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = cnn(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(xb)

    cnn.eval()
    predictions = []
    actual = []

    with torch.no_grad():
        for xb, yb in val_loader:
            logits = cnn(xb.to(device))
            predictions.extend(logits.argmax(dim=1).cpu().numpy())
            actual.extend(yb.numpy())

    val_f1 = f1_score(actual, predictions, average="macro", zero_division=0)
    val_acc = accuracy_score(actual, predictions)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {running_loss / len(train_dataset):.4f} | "
        f"Val Accuracy: {val_acc:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1 + 1e-5:
        best_f1 = val_f1
        best_state = copy.deepcopy(cnn.state_dict())
        best_epoch = epoch
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        print("Early stopping")
        break

cnn.load_state_dict(best_state)

model_path = output_dir / "karsl_1d_cnn_10_classes_v1.pt"

torch.save({
    "model_state_dict": cnn.cpu().state_dict(),
    "classes": classes,
    "input_shape": [32, 150],
    "architecture": "SignCNN",
    "preprocessing": "preprocess_pose_v1",
    "best_epoch": best_epoch,
    "validation_macro_f1": float(best_f1)
}, model_path)

print("\n=== FINAL CNN RESULT ===")
print("Best epoch:", best_epoch)
print("Best Validation Macro F1:", round(best_f1, 4))
print("Model saved:", model_path)
print("New downloads: NO")
print("GitHub modified: NO")


In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")

checkpoint = torch.load(
    output_dir / "karsl_1d_cnn_10_classes_v1.pt",
    map_location="cpu",
    weights_only=True
)

with np.load(output_dir / "pose_10_classes_features_v2.npz") as data:
    X_val = data["X_val"].reshape(-1, 32, 150)
    y_val = data["y_val"]

classes = checkpoint["classes"]

loaded_model = SignCNN(len(classes))
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

inputs = torch.tensor(
    X_val, dtype=torch.float32
).transpose(1, 2)

with torch.no_grad():
    logits = loaded_model(inputs)
    indices = logits.argmax(dim=1).numpy()

predictions = np.array([classes[i] for i in indices])

accuracy = accuracy_score(y_val, predictions)
macro_f1 = f1_score(
    y_val, predictions, average="macro", zero_division=0
)

report = classification_report(
    y_val,
    predictions,
    labels=classes,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    output_dir / "cnn_10_classes_validation_report.csv"
)

print("=== SAVED CNN VERIFICATION ===")
print("Model loaded: YES")
print("Best epoch:", checkpoint["best_epoch"])
print("Input shape:", X_val.shape)
print("Validation samples:", len(y_val))
print("Accuracy:", round(accuracy * 100, 2), "%")
print("Macro F1:", round(macro_f1, 4))

assert abs(macro_f1 - checkpoint["validation_macro_f1"]) < 1e-6
assert np.isfinite(logits.numpy()).all()

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_val,
    predictions,
    labels=classes,
    zero_division=0
))

print("\nSaved model verification: PASS")
print("Training repeated: NO")
print("New downloads: NO")
print("GitHub modified: NO")


## 4. Historical 30-class feature and model experiments


In [ ]:
import numpy as np

BODY_INDICES = [11, 12, 13, 14, 15, 16, 23, 24]
HAND_INDICES = list(range(501, 543))
LANDMARK_INDICES = BODY_INDICES + HAND_INDICES

TARGET_FRAMES = 32
CONFIDENCE_THRESHOLD = 0.5
FEATURE_DIM = len(LANDMARK_INDICES) * 3

def preprocess_pose(pose_object):
    data = np.asarray(pose_object.body.data, dtype=np.float32)
    confidence = np.asarray(pose_object.body.confidence, dtype=np.float32)

    if data.ndim != 4 or data.shape[1:] != (1, 543, 3):
        raise ValueError(f"Unexpected pose shape: {data.shape}")

    if confidence.shape != data.shape[:3]:
        raise ValueError(f"Unexpected confidence shape: {confidence.shape}")

    if data.shape[0] == 0:
        raise ValueError("Empty pose sequence")

    coordinates = data[:, 0, :, :2]
    scores = confidence[:, 0, :]

    shoulder_valid = (
        (scores[:, 11] > CONFIDENCE_THRESHOLD)
        & (scores[:, 12] > CONFIDENCE_THRESHOLD)
        & np.isfinite(coordinates[:, 11]).all(axis=1)
        & np.isfinite(coordinates[:, 12]).all(axis=1)
    )

    if not shoulder_valid.any():
        raise ValueError("No valid shoulder reference found")

    left_shoulder = coordinates[shoulder_valid, 11]
    right_shoulder = coordinates[shoulder_valid, 12]

    centers = (left_shoulder + right_shoulder) / 2
    widths = np.linalg.norm(left_shoulder - right_shoulder, axis=1)

    center = np.median(centers, axis=0)
    valid_widths = widths[widths > 1e-6]

    if len(valid_widths) == 0:
        raise ValueError("Invalid shoulder width")

    scale = float(np.median(valid_widths))

    selected = coordinates[:, LANDMARK_INDICES, :]
    selected_scores = scores[:, LANDMARK_INDICES]

    valid = (
        (selected_scores > CONFIDENCE_THRESHOLD)
        & np.isfinite(selected).all(axis=2)
    )

    normalized = (selected - center) / scale
    normalized = np.clip(normalized, -5.0, 5.0)
    normalized = np.where(valid[..., None], normalized, 0.0)

    features = np.concatenate(
        [normalized, valid[..., None].astype(np.float32)],
        axis=2
    )

    frame_indices = np.rint(
        np.linspace(0, len(features) - 1, TARGET_FRAMES)
    ).astype(int)

    features = features[frame_indices]
    features = features.reshape(TARGET_FRAMES, FEATURE_DIM)

    return features.astype(np.float32)

print("Preprocessing: READY")
print("Output shape: (32, 150)")


In [ ]:
%pip install -q pose-format
from pose_format import Pose
print("Pose format: READY")


In [ ]:
import numpy as np
import pandas as pd
import kagglehub
from pathlib import Path
from pose_format import Pose
from contextlib import redirect_stdout, redirect_stderr
import io

assert "pose_folds" in globals(), "Run the Pose metadata cell first"
assert "preprocess_pose" in globals(), "Run the preprocessing function cell first"

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
output_dir.mkdir(parents=True, exist_ok=True)

cache_path = output_dir / "pose_30_classes_features_v1.npz"
preprocessing_version = "pose_v1_32x50_xy_visibility_shoulder_conf05"

fold = pose_folds[pose_folds["fold"] == "fold_1"].copy()

counts = fold.groupby(["sign_id", "split"]).size().unstack(fill_value=0)
counts = counts.reindex(columns=["train", "validation"], fill_value=0)

eligible = counts[
    (counts["train"] >= 12) &
    (counts["validation"] >= 6)
]

classes_30 = eligible.index[:30].tolist()
assert len(classes_30) == 30

def choose_rows(split, n):
    subset = fold[
        (fold["split"] == split) &
        (fold["sign_id"].isin(classes_30))
    ]
    return subset.groupby("sign_id", group_keys=False).sample(
        n=n,
        random_state=42
    )

train_30 = choose_rows("train", 12)
val_30 = choose_rows("validation", 6)

assert set(train_30["signer_id"]).isdisjoint(
    set(val_30["signer_id"])
)

assert set(train_30["pose_file"]).isdisjoint(
    set(val_30["pose_file"])
)

rows = pd.concat(
    [train_30, val_30],
    ignore_index=True
)

paths = rows["pose_file"].astype(str).to_numpy(dtype=str)

if cache_path.is_file():
    with np.load(cache_path, allow_pickle=True) as saved:
        saved_paths = saved["paths"].astype(str)
        saved_version = str(saved["preprocessing_version"].item())

        assert np.array_equal(saved_paths, paths), (
            "Saved cache has different sample paths"
        )

        assert saved_version == preprocessing_version, (
            "Saved cache uses a different preprocessing version"
        )

        X_30 = saved["X"].copy()
        done = saved["done"].astype(bool).copy()

    assert X_30.shape == (len(rows), 32, 150)
    assert done.shape == (len(rows),)

    print("Resuming saved preprocessing")

else:
    X_30 = np.zeros(
        (len(rows), 32, 150),
        dtype=np.float32
    )
    done = np.zeros(len(rows), dtype=bool)

def save_progress():
    temporary = cache_path.with_suffix(".tmp.npz")

    np.savez_compressed(
        temporary,
        X=X_30,
        done=done,
        paths=paths,
        preprocessing_version=np.asarray(preprocessing_version)
    )

    temporary.replace(cache_path)

print("=== KARSL 30-CLASS PREPROCESSING ===")
print("Classes:", len(classes_30))
print("Train:", len(train_30))
print("Validation:", len(val_30))
print("Already completed:", int(done.sum()))
print("Remaining:", int((~done).sum()))

try:
    for i, row in enumerate(rows.itertuples(index=False)):
        if done[i]:
            continue

        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            file_path = Path(
                kagglehub.dataset_download(
                    "tfmohamedyahia/kars502-pose-75",
                    path="content/kars502-pose-75k/" + row.pose_file
                )
            )

        with file_path.open("rb") as f:
            sample = Pose.read(f.read())

        features = preprocess_pose(sample)

        assert features.shape == (32, 150)
        assert np.isfinite(features).all()

        X_30[i] = features
        done[i] = True

        if int(done.sum()) % 40 == 0:
            save_progress()
            print(f"Completed: {int(done.sum())}/{len(rows)}")

finally:
    save_progress()

assert done.all(), "Preprocessing is incomplete"
assert np.isfinite(X_30).all()

print("\n=== FINAL STATUS ===")
print("Processed:", int(done.sum()), "/", len(rows))
print("Feature shape:", X_30.shape)
print("Classes:", classes_30)
print("Cache:", cache_path)
print("Status: PASS")
print("Training repeated: NO")
print("GitHub modified: NO")


In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import DataLoader, TensorDataset

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
cache_path = output_dir / "pose_30_classes_features_v1.npz"
model_path = output_dir / "karsl_1d_cnn_30_classes_v1.pt"

assert cache_path.is_file()
assert "rows" in globals()
assert "train_30" in globals()
assert "val_30" in globals()

with np.load(cache_path, allow_pickle=True) as saved:
    X = saved["X"].astype(np.float32)
    done = saved["done"].astype(bool)
    saved_paths = saved["paths"].astype(str)
    preprocessing_version = str(saved["preprocessing_version"].item())

assert preprocessing_version == "pose_v1_32x50_xy_visibility_shoulder_conf05"
assert X.shape == (540, 32, 150)
assert done.all()
assert np.isfinite(X).all()
assert np.array_equal(saved_paths, rows["pose_file"].astype(str).to_numpy())

train_count = len(train_30)
val_count = len(val_30)

assert train_count == 360
assert val_count == 180

classes = sorted(rows["sign_id"].unique().tolist())
assert len(classes) == 30

label_to_index = {label: i for i, label in enumerate(classes)}

X_train = X[:train_count]
X_val = X[train_count:]

y_train = np.array(
    [label_to_index[v] for v in train_30["sign_id"]],
    dtype=np.int64
)

y_val = np.array(
    [label_to_index[v] for v in val_30["sign_id"]],
    dtype=np.int64
)

assert set(train_30["signer_id"]).isdisjoint(set(val_30["signer_id"]))
assert set(train_30["pose_file"]).isdisjoint(set(val_30["pose_file"]))

train_data = TensorDataset(
    torch.from_numpy(X_train).transpose(1, 2),
    torch.from_numpy(y_train)
)

val_data = TensorDataset(
    torch.from_numpy(X_val).transpose(1, 2),
    torch.from_numpy(y_val)
)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=64,
    shuffle=False
)

class SignCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(150, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SignCNN(len(classes)).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.001
)

criterion = nn.CrossEntropyLoss()

best_f1 = -1.0
best_epoch = 0
patience = 8
wait = 0
history = []

print("=== KARSL 30-CLASS CNN TRAINING ===")
print("Device:", device)
print("Classes:", len(classes))
print("Train samples:", train_count)
print("Validation samples:", val_count)
print("Input shape:", X_train.shape[1:])

for epoch in range(1, 51):
    model.train()
    total_loss = 0.0
    train_predictions = []
    train_actual = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)
        train_predictions.extend(logits.argmax(1).detach().cpu().numpy())
        train_actual.extend(yb.cpu().numpy())

    train_accuracy = accuracy_score(train_actual, train_predictions)

    model.eval()
    val_predictions = []
    val_actual = []

    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb.to(device))
            val_predictions.extend(logits.argmax(1).cpu().numpy())
            val_actual.extend(yb.numpy())

    val_accuracy = accuracy_score(val_actual, val_predictions)
    val_f1 = f1_score(
        val_actual,
        val_predictions,
        average="macro",
        zero_division=0
    )

    history.append({
        "epoch": epoch,
        "train_loss": total_loss / train_count,
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_f1
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {total_loss / train_count:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1 + 1e-5:
        best_f1 = val_f1
        best_epoch = epoch
        wait = 0

        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        torch.save({
            "model_state_dict": best_state,
            "classes": classes,
            "input_shape": [32, 150],
            "architecture": "SignCNN",
            "preprocessing_version": preprocessing_version,
            "best_epoch": best_epoch,
            "validation_macro_f1": float(best_f1)
        }, model_path)

    else:
        wait += 1

    if wait >= patience:
        print("Early stopping")
        break

pd.DataFrame(history).to_csv(
    output_dir / "cnn_30_classes_training_history.csv",
    index=False
)

checkpoint = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

verified_model = SignCNN(len(classes))
verified_model.load_state_dict(checkpoint["model_state_dict"])
verified_model.eval()

with torch.no_grad():
    final_logits = verified_model(
        torch.from_numpy(X_val).transpose(1, 2)
    )
    final_predictions = final_logits.argmax(1).numpy()

final_accuracy = accuracy_score(y_val, final_predictions)
final_f1 = f1_score(
    y_val,
    final_predictions,
    average="macro",
    zero_division=0
)

assert abs(final_f1 - best_f1) < 1e-6
assert torch.isfinite(final_logits).all()

report = classification_report(
    y_val,
    final_predictions,
    labels=list(range(len(classes))),
    target_names=[str(label) for label in classes],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    output_dir / "cnn_30_classes_validation_report.csv"
)

print("\n=== FINAL 30-CLASS CNN RESULT ===")
print("Best epoch:", best_epoch)
print("Validation Accuracy:", round(final_accuracy * 100, 2), "%")
print("Validation Macro F1:", round(final_f1, 4))
print("Saved model verified: PASS")
print("Model path:", model_path)
print("Training history saved: YES")
print("Validation report saved: YES")
print("New downloads: NO")
print("GitHub modified: NO")


In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

output_dir = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
cache_path = output_dir / "pose_30_classes_features_v1.npz"

with np.load(cache_path, allow_pickle=True) as saved:
    X = saved["X"].astype(np.float32)
    done = saved["done"].astype(bool)
    paths = saved["paths"].astype(str)
    version = str(saved["preprocessing_version"].item())

assert X.shape == (540, 32, 150)
assert done.all()
assert version == "pose_v1_32x50_xy_visibility_shoulder_conf05"

fold = pose_folds[pose_folds["fold"] == "fold_1"].copy()
assert not fold["pose_file"].duplicated().any()

fold = fold.set_index("pose_file")
details = fold.loc[paths]

assert (details["split"].iloc[:360] == "train").all()
assert (details["split"].iloc[360:] == "validation").all()

train_signers = set(details["signer_id"].iloc[:360])
val_signers = set(details["signer_id"].iloc[360:])

assert train_signers.isdisjoint(val_signers)

X_train = X[:360].reshape(360, -1)
X_val = X[360:].reshape(180, -1)

y_train = details["sign_id"].iloc[:360].to_numpy()
y_val = details["sign_id"].iloc[360:].to_numpy()

models = {
    "SVM C=1": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=1, gamma="scale")
    ),
    "SVM C=10": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=10, gamma="scale")
    )
}

results = []
best_model = None
best_f1 = -1
best_name = None

for name, model in models.items():
    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)

    train_acc = accuracy_score(y_train, train_predictions)
    val_acc = accuracy_score(y_val, val_predictions)
    val_f1 = f1_score(
        y_val,
        val_predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Train Accuracy": round(train_acc, 4),
        "Validation Accuracy": round(val_acc, 4),
        "Validation Macro F1": round(val_f1, 4)
    })

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model = model
        best_name = name

results.append({
    "Model": "CNN",
    "Train Accuracy": np.nan,
    "Validation Accuracy": 0.1444,
    "Validation Macro F1": 0.1135
})

comparison = pd.DataFrame(results)

comparison.to_csv(
    output_dir / "30_classes_model_comparison.csv",
    index=False
)

model_path = output_dir / "karsl_svm_30_classes_best.joblib"
joblib.dump(best_model, model_path)

print("=== KARSL 30-CLASS MODEL COMPARISON ===")
print(comparison.to_string(index=False))

print("\nBest SVM:", best_name)
print("Saved:", model_path)
print("New downloads: NO")
print("CNN overwritten: NO")
print("Test set used: NO")
print("GitHub modified: NO")


In [ ]:
import numpy as np
import pandas as pd
import kagglehub
from pathlib import Path
from pose_format import Pose

assert "pose_folds" in globals()
assert callable(preprocess_pose)

folder = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
old_path = folder / "pose_30_classes_features_v1.npz"
new_path = folder / "pose_30_classes_features_v2.npz"

version = "pose_v1_32x50_xy_visibility_shoulder_conf05"

fold = pose_folds[pose_folds["fold"] == "fold_1"].copy()
classes = list(range(1, 31))

def select_rows(split, count):
    subset = fold[
        (fold["sign_id"].isin(classes)) &
        (fold["split"] == split)
    ]

    counts = subset.groupby("sign_id").size()
    assert all(counts.get(c, 0) >= count for c in classes)

    return subset.groupby(
        "sign_id", group_keys=False
    ).sample(n=count, random_state=42)

train_rows = select_rows("train", 30)
val_rows = select_rows("validation", 15)

assert set(train_rows["signer_id"]).isdisjoint(
    set(val_rows["signer_id"])
)

rows_v2 = pd.concat([train_rows, val_rows], ignore_index=True)
paths_v2 = rows_v2["pose_file"].astype(str).to_numpy(dtype=str)

assert len(paths_v2) == len(set(paths_v2))

old_features = {}

with np.load(old_path, allow_pickle=True) as old:
    assert str(old["preprocessing_version"].item()) == version

    for path, features, complete in zip(
        old["paths"].astype(str),
        old["X"],
        old["done"]
    ):
        if complete:
            old_features[path] = features.copy()

if new_path.is_file():
    with np.load(new_path, allow_pickle=True) as saved:
        assert np.array_equal(
            saved["paths"].astype(str), paths_v2
        )
        assert str(saved["preprocessing_version"].item()) == version

        X_v2 = saved["X"].copy()
        done_v2 = saved["done"].astype(bool).copy()

else:
    X_v2 = np.zeros(
        (len(rows_v2), 32, 150),
        dtype=np.float32
    )
    done_v2 = np.zeros(len(rows_v2), dtype=bool)

reused = 0

for i, path in enumerate(paths_v2):
    if not done_v2[i] and path in old_features:
        X_v2[i] = old_features[path]
        done_v2[i] = True
        reused += 1

def save_v2():
    temporary = new_path.with_suffix(".tmp.npz")

    np.savez_compressed(
        temporary,
        X=X_v2,
        done=done_v2,
        paths=paths_v2,
        preprocessing_version=np.asarray(version)
    )

    temporary.replace(new_path)

save_v2()

print("=== KARSL 30-CLASS DATA EXPANSION ===")
print("Classes:", len(classes))
print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Reused from old cache:", reused)
print("Already completed:", int(done_v2.sum()))
print("Remaining:", int((~done_v2).sum()))

try:
    for i, row in enumerate(rows_v2.itertuples(index=False)):
        if done_v2[i]:
            continue

        file_path = Path(
            kagglehub.dataset_download(
                "tfmohamedyahia/kars502-pose-75",
                path="content/kars502-pose-75k/" + row.pose_file
            )
        )

        with file_path.open("rb") as f:
            sample = Pose.read(f.read())

        features = preprocess_pose(sample)

        assert features.shape == (32, 150)
        assert np.isfinite(features).all()

        X_v2[i] = features
        done_v2[i] = True

        if int(done_v2.sum()) % 50 == 0:
            save_v2()
            print(f"Completed: {int(done_v2.sum())}/{len(rows_v2)}")

finally:
    save_v2()

assert done_v2.all()
assert X_v2.shape == (1350, 32, 150)
assert np.isfinite(X_v2).all()

print("\n=== FINAL DATA STATUS ===")
print("Processed:", int(done_v2.sum()), "/", len(rows_v2))
print("Feature shape:", X_v2.shape)
print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Status: PASS")
print("Old cache preserved: YES")
print("Existing models modified: NO")
print("Test set used: NO")
print("GitHub modified: NO")


In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import TensorDataset, DataLoader

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

folder = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
new_cache = folder / "pose_30_classes_features_v2.npz"
old_cache = folder / "pose_30_classes_features_v1.npz"
model_path = folder / "karsl_1d_cnn_30_classes_v2.pt"
history_path = folder / "cnn_30_classes_v2_history.csv"

with np.load(new_cache, allow_pickle=True) as saved:
    X = saved["X"].astype(np.float32)
    done = saved["done"].astype(bool)
    paths = saved["paths"].astype(str)
    version = str(saved["preprocessing_version"].item())

assert X.shape == (1350, 32, 150)
assert done.all()
assert np.isfinite(X).all()
assert version == "pose_v1_32x50_xy_visibility_shoulder_conf05"
assert len(set(paths)) == len(paths)

fold = pose_folds[pose_folds["fold"] == "fold_1"].copy()
assert not fold["pose_file"].duplicated().any()

details = fold.set_index("pose_file").loc[paths]

assert (details["split"].iloc[:900] == "train").all()
assert (details["split"].iloc[900:] == "validation").all()

train_signers = set(details["signer_id"].iloc[:900])
val_signers = set(details["signer_id"].iloc[900:])

assert train_signers.isdisjoint(val_signers)

classes = sorted(details["sign_id"].unique().astype(int).tolist())
assert len(classes) == 30

label_map = {label: i for i, label in enumerate(classes)}

y = np.array(
    [label_map[int(label)] for label in details["sign_id"]],
    dtype=np.int64
)

X_train = X[:900]
X_val = X[900:]
y_train = y[:900]
y_val = y[900:]

assert np.bincount(y_train, minlength=30).tolist() == [30] * 30
assert np.bincount(y_val, minlength=30).tolist() == [15] * 30

train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train).transpose(1, 2),
        torch.from_numpy(y_train)
    ),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_val).transpose(1, 2),
        torch.from_numpy(y_val)
    ),
    batch_size=64
)

class SignCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(150, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SignCNN(len(classes)).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.001
)

criterion = nn.CrossEntropyLoss()

best_f1 = -1.0
best_epoch = 0
wait = 0
patience = 8
history = []

print("=== KARSL 30-CLASS CNN V2 TRAINING ===")
print("Device:", device)
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Classes:", len(classes))

for epoch in range(1, 51):
    model.train()
    total_loss = 0.0
    train_correct = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)
        train_correct += (logits.argmax(1) == yb).sum().item()

    model.eval()
    predictions = []
    actual = []

    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb.to(device))
            predictions.extend(logits.argmax(1).cpu().numpy())
            actual.extend(yb.numpy())

    train_acc = train_correct / len(X_train)
    val_acc = accuracy_score(actual, predictions)
    val_f1 = f1_score(
        actual,
        predictions,
        average="macro",
        zero_division=0
    )

    history.append({
        "epoch": epoch,
        "train_loss": total_loss / len(X_train),
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1
    })

    pd.DataFrame(history).to_csv(history_path, index=False)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1 + 1e-5:
        best_f1 = val_f1
        best_epoch = epoch
        wait = 0

        state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        torch.save({
            "model_state_dict": state,
            "classes": classes,
            "architecture": "SignCNN",
            "input_shape": [32, 150],
            "preprocessing_version": version,
            "best_epoch": best_epoch,
            "validation_macro_f1": float(best_f1)
        }, model_path)
    else:
        wait += 1

    if wait >= patience:
        print("Early stopping")
        break

checkpoint = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

verified = SignCNN(len(classes))
verified.load_state_dict(checkpoint["model_state_dict"])
verified.eval()

def evaluate_saved(features, labels):
    with torch.no_grad():
        logits = verified(
            torch.from_numpy(features).transpose(1, 2)
        )
        predictions = logits.argmax(1).numpy()

    return (
        accuracy_score(labels, predictions),
        f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

val_acc, val_f1 = evaluate_saved(X_val, y_val)
assert abs(val_f1 - best_f1) < 1e-6

with np.load(old_cache, allow_pickle=True) as saved:
    old_paths = saved["paths"].astype(str)
    old_done = saved["done"].astype(bool)

assert old_done.all()
assert len(old_paths) == 540

path_to_index = {path: i for i, path in enumerate(paths)}
old_val_paths = old_paths[360:]

assert set(old_val_paths).issubset(set(paths[900:]))

old_indices = np.array([
    path_to_index[path] for path in old_val_paths
])

old_val_acc, old_val_f1 = evaluate_saved(
    X[old_indices],
    y[old_indices]
)

comparison = pd.DataFrame([
    {
        "Model": "CNN V1",
        "Train Samples": 360,
        "Validation Samples": 180,
        "Accuracy": 0.1444,
        "Macro F1": 0.1135
    },
    {
        "Model": "CNN V2 on old validation",
        "Train Samples": 900,
        "Validation Samples": 180,
        "Accuracy": old_val_acc,
        "Macro F1": old_val_f1
    },
    {
        "Model": "CNN V2 on expanded validation",
        "Train Samples": 900,
        "Validation Samples": 450,
        "Accuracy": val_acc,
        "Macro F1": val_f1
    }
])

comparison.to_csv(
    folder / "cnn_30_classes_v2_comparison.csv",
    index=False
)

print("\n=== FINAL CNN V2 RESULT ===")
print("Best epoch:", best_epoch)
print("Validation Accuracy:", round(val_acc * 100, 2), "%")
print("Validation Macro F1:", round(val_f1, 4))

print("\n=== FAIR COMPARISON ===")
print(comparison.round(4).to_string(index=False))

print("\nSaved model verified: PASS")
print("Model:", model_path)
print("New downloads: NO")
print("Test set used: NO")
print("GitHub modified: NO")


In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score, f1_score

folder = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
model_path = folder / "karsl_1d_cnn_30_classes_v2.pt"
cache_path = folder / "pose_30_classes_features_v2.npz"

checkpoint = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

classes = [int(c) for c in checkpoint["classes"]]

model = SignCNN(len(classes))
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with np.load(cache_path, allow_pickle=True) as saved:
    X = saved["X"].astype(np.float32)
    paths = saved["paths"].astype(str)
    done = saved["done"].astype(bool)
    version = str(saved["preprocessing_version"].item())

assert X.shape == (1350, 32, 150)
assert done.all()
assert checkpoint["preprocessing_version"] == version

def predict_sign_features(features, top_k=3):
    features = np.asarray(features, dtype=np.float32)

    if features.shape != (32, 150):
        raise ValueError("Expected features with shape (32, 150)")

    if not np.isfinite(features).all():
        raise ValueError("Features contain invalid values")

    inputs = torch.from_numpy(
        features[None, ...]
    ).transpose(1, 2)

    with torch.no_grad():
        logits = model(inputs)
        probabilities = torch.softmax(logits, dim=1)[0].numpy()

    indices = np.argsort(probabilities)[::-1][:top_k]

    return [
        {
            "sign_id": classes[int(i)],
            "score": round(float(probabilities[i]), 4)
        }
        for i in indices
    ]

def predict_pose_file(pose_path, top_k=3):
    from pose_format import Pose

    with open(pose_path, "rb") as f:
        pose = Pose.read(f.read())

    features = preprocess_pose(pose)

    return predict_sign_features(features, top_k=top_k)

X_val = X[900:]
val_paths = paths[900:]

fold = pose_folds[pose_folds["fold"] == "fold_1"].copy()
details = fold.set_index("pose_file").loc[val_paths]

assert (details["split"] == "validation").all()

y_true = details["sign_id"].astype(int).to_numpy()

predictions = []
scores = []

for features in X_val:
    result = predict_sign_features(features, top_k=1)[0]
    predictions.append(result["sign_id"])
    scores.append(result["score"])

accuracy = accuracy_score(y_true, predictions)
macro_f1 = f1_score(
    y_true,
    predictions,
    average="macro",
    zero_division=0
)

report = classification_report(
    y_true,
    predictions,
    labels=classes,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    folder / "cnn_30_classes_v2_per_class_report.csv"
)

prediction_df = pd.DataFrame({
    "pose_file": val_paths,
    "true_sign_id": y_true,
    "predicted_sign_id": predictions,
    "top1_score": scores,
    "correct": y_true == np.array(predictions)
})

prediction_df.to_csv(
    folder / "cnn_30_classes_v2_predictions.csv",
    index=False
)

print("=== PREDICTION FUNCTION TEST ===")
print("Input:", X_val[0].shape)
print("Actual Sign ID:", int(y_true[0]))
print("Top 3 Predictions:", predict_sign_features(X_val[0]))

print("\n=== MODEL EVALUATION ===")
print("Validation Accuracy:", round(accuracy * 100, 2), "%")
print("Validation Macro F1:", round(macro_f1, 4))
print("Samples evaluated:", len(X_val))

print("\n=== LOWEST F1 CLASSES ===")
print(
    report_df.loc[
        [str(c) for c in classes],
        ["precision", "recall", "f1-score", "support"]
    ].sort_values("f1-score").head(5).round(4).to_string()
)

assert abs(macro_f1 - checkpoint["validation_macro_f1"]) < 1e-4

print("\n=== FINAL STATUS ===")
print("Prediction function: PASS")
print("Saved model verification: PASS")
print("Per-class report saved: YES")
print("Predictions CSV saved: YES")
print("Training repeated: NO")
print("Downloads: NO")
print("Test set used: NO")
print("GitHub modified: NO")


## 5. Historical CNN inference, packaging, and test evaluation

The source is retained for traceability. These cells have no saved outputs in this export. Previously recorded test results must not guide new model selection.


In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

base = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
package = base / "package"
package.mkdir(parents=True, exist_ok=True)

module_path = package / "karsl_inference.py"

source = r'''
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

BODY_INDICES = [11, 12, 13, 14, 15, 16, 23, 24]
HAND_INDICES = list(range(501, 543))
LANDMARK_INDICES = BODY_INDICES + HAND_INDICES
PREPROCESSING_VERSION = "pose_v1_32x50_xy_visibility_shoulder_conf05"


def preprocess_pose(pose_object):
    data = np.asarray(pose_object.body.data, dtype=np.float32)
    confidence = np.asarray(
        pose_object.body.confidence,
        dtype=np.float32
    )

    if data.ndim != 4 or data.shape[1:] != (1, 543, 3):
        raise ValueError(f"Unexpected pose shape: {data.shape}")

    if confidence.shape != data.shape[:3]:
        raise ValueError("Unexpected confidence shape")

    if data.shape[0] == 0:
        raise ValueError("Empty pose sequence")

    coordinates = data[:, 0, :, :2]
    scores = confidence[:, 0, :]

    shoulder_valid = (
        (scores[:, 11] > 0.5)
        & (scores[:, 12] > 0.5)
        & np.isfinite(coordinates[:, 11]).all(axis=1)
        & np.isfinite(coordinates[:, 12]).all(axis=1)
    )

    if not shoulder_valid.any():
        raise ValueError("No valid shoulder reference")

    left = coordinates[shoulder_valid, 11]
    right = coordinates[shoulder_valid, 12]

    centers = (left + right) / 2
    widths = np.linalg.norm(left - right, axis=1)

    center = np.median(centers, axis=0)
    valid_widths = widths[widths > 1e-6]

    if len(valid_widths) == 0:
        raise ValueError("Invalid shoulder width")

    scale = float(np.median(valid_widths))

    selected = coordinates[:, LANDMARK_INDICES, :]
    selected_scores = scores[:, LANDMARK_INDICES]

    valid = (
        (selected_scores > 0.5)
        & np.isfinite(selected).all(axis=2)
    )

    normalized = (selected - center) / scale
    normalized = np.clip(normalized, -5.0, 5.0)
    normalized = np.where(valid[..., None], normalized, 0.0)

    features = np.concatenate(
        [normalized, valid[..., None].astype(np.float32)],
        axis=2
    )

    indices = np.rint(
        np.linspace(0, len(features) - 1, 32)
    ).astype(int)

    return features[indices].reshape(32, 150).astype(np.float32)


class SignCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(150, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)


class KArSLPredictor:
    def __init__(self, model_path):
        checkpoint = torch.load(
            Path(model_path),
            map_location="cpu",
            weights_only=True
        )

        if checkpoint["preprocessing_version"] != PREPROCESSING_VERSION:
            raise ValueError("Preprocessing version mismatch")

        if checkpoint["input_shape"] != [32, 150]:
            raise ValueError("Input shape mismatch")

        self.classes = [int(x) for x in checkpoint["classes"]]
        self.model = SignCNN(len(self.classes))
        self.model.load_state_dict(
            checkpoint["model_state_dict"]
        )
        self.model.eval()

    def predict_features(self, features, top_k=3):
        features = np.asarray(features, dtype=np.float32)

        if features.shape != (32, 150):
            raise ValueError("Expected shape (32, 150)")

        if not np.isfinite(features).all():
            raise ValueError("Invalid feature values")

        if not isinstance(top_k, int) or not 1 <= top_k <= len(self.classes):
            raise ValueError("Invalid top_k")

        inputs = torch.from_numpy(
            features[None, ...]
        ).transpose(1, 2)

        with torch.no_grad():
            logits = self.model(inputs)
            probabilities = torch.softmax(
                logits, dim=1
            )[0].numpy()

        indices = np.argsort(probabilities)[::-1][:top_k]

        return [
            {
                "sign_id": self.classes[int(i)],
                "score": round(float(probabilities[i]), 4)
            }
            for i in indices
        ]

    def predict_pose(self, pose_object, top_k=3):
        features = preprocess_pose(pose_object)
        return self.predict_features(features, top_k=top_k)

    def predict_pose_file(self, pose_path, top_k=3):
        from pose_format import Pose

        with open(pose_path, "rb") as f:
            pose = Pose.read(f.read())

        return self.predict_pose(pose, top_k=top_k)
'''

module_path.write_text(source.strip() + "\n", encoding="utf-8")

spec = importlib.util.spec_from_file_location(
    "karsl_inference",
    module_path
)

inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference)

predictor = inference.KArSLPredictor(
    base / "karsl_1d_cnn_30_classes_v2.pt"
)

with np.load(
    base / "pose_30_classes_features_v2.npz",
    allow_pickle=True
) as saved:
    X = saved["X"].astype(np.float32)
    paths = saved["paths"].astype(str)
    done = saved["done"].astype(bool)

assert done.all()
assert X.shape == (1350, 32, 150)

saved_predictions = pd.read_csv(
    base / "cnn_30_classes_v2_predictions.csv"
)

assert np.array_equal(
    saved_predictions["pose_file"].astype(str).to_numpy(),
    paths[900:]
)

for i in range(5):
    result = predictor.predict_features(X[900 + i], top_k=3)
    expected = saved_predictions.iloc[i]

    assert result[0]["sign_id"] == int(expected["predicted_sign_id"])
    assert abs(result[0]["score"] - float(expected["top1_score"])) < 0.0002

invalid_input_rejected = False

try:
    predictor.predict_features(np.zeros((32, 149), dtype=np.float32))
except ValueError:
    invalid_input_rejected = True

assert invalid_input_rejected

print("=== INDEPENDENT INFERENCE MODULE ===")
print("Module created:", module_path)
print("Model loading: PASS")
print("Cached prediction checks: PASS")
print("Input validation: PASS")
print("Top 3 Predictions:", predictor.predict_features(X[900]))
print("Existing models modified: NO")
print("GitHub modified: NO")

local_pose_path = (
    Path("/root/.cache/kagglehub/datasets")
    / "tfmohamedyahia/kars502-pose-75/versions/1"
    / "content/kars502-pose-75k"
    / paths[900]
)

if local_pose_path.is_file():
    from pose_format import Pose

    with local_pose_path.open("rb") as f:
        pose = Pose.read(f.read())

    independent_features = inference.preprocess_pose(pose)
    cached_features = X[900]

    assert np.allclose(
        independent_features,
        cached_features,
        atol=1e-6,
        rtol=1e-6
    ), "Preprocessing does not match saved features"

    if "preprocess_pose" in globals():
        original_features = preprocess_pose(pose)
        assert np.allclose(
            independent_features,
            original_features,
            atol=1e-6,
            rtol=1e-6
        ), "Independent preprocessing differs from notebook"

    print("Pose preprocessing compatibility: PASS")
else:
    print("Pose preprocessing compatibility: NOT CHECKED")

print("\n=== FINAL STATUS ===")
print("Independent module: CREATED")
print("Saved predictions: VERIFIED")
print("No training or downloads performed")


In [ ]:
import numpy as np
import kagglehub
from pathlib import Path
from pose_format import Pose

base = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
dataset_root = Path(
    "/root/.cache/kagglehub/datasets/"
    "tfmohamedyahia/kars502-pose-75/versions/1/"
    "content/kars502-pose-75k"
)

assert "predictor" in globals()
assert "inference" in globals()

with np.load(
    base / "pose_30_classes_features_v2.npz",
    allow_pickle=True
) as saved:
    X = saved["X"].astype(np.float32)
    paths = saved["paths"].astype(str)
    done = saved["done"].astype(bool)

assert done.all()

sample_index = None
pose_path = None
downloaded = False

for i in range(900, len(paths)):
    candidate = dataset_root / paths[i]

    if candidate.is_file():
        sample_index = i
        pose_path = candidate
        break

if pose_path is None:
    sample_index = 900

    pose_path = Path(
        kagglehub.dataset_download(
            "tfmohamedyahia/kars502-pose-75",
            path="content/kars502-pose-75k/" + paths[sample_index]
        )
    )

    downloaded = True

with pose_path.open("rb") as f:
    pose = Pose.read(f.read())

new_features = inference.preprocess_pose(pose)
saved_features = X[sample_index]

maximum_difference = float(
    np.max(np.abs(new_features - saved_features))
)

assert np.allclose(
    new_features,
    saved_features,
    atol=1e-6,
    rtol=1e-6
), f"Preprocessing mismatch: {maximum_difference}"

feature_prediction = predictor.predict_features(
    saved_features,
    top_k=3
)

file_prediction = predictor.predict_pose_file(
    pose_path,
    top_k=3
)

assert feature_prediction == file_prediction

print("=== KARSL END-TO-END TEST ===")
print("Pose file:", paths[sample_index])
print("Sample index:", sample_index)
print("Pose loaded: PASS")
print("Feature shape:", new_features.shape)
print("Preprocessing compatibility: PASS")
print("Maximum feature difference:", maximum_difference)
print("Prediction consistency: PASS")
print("Top 3 Predictions:", file_prediction)
print("Downloaded one file:", downloaded)
print("Training repeated: NO")
print("Existing models modified: NO")
print("GitHub modified: NO")


In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

base = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
delivery = base / "delivery_30class"
ml_dir = delivery / "karsl" / "ml"
models_dir = ml_dir / "models"
reports_dir = ml_dir / "reports"
tests_dir = delivery / "tests"

for folder in [models_dir, reports_dir, tests_dir]:
    folder.mkdir(parents=True, exist_ok=True)

source_module = base / "package" / "karsl_inference.py"
source_model = base / "karsl_1d_cnn_30_classes_v2.pt"

assert source_module.is_file()
assert source_model.is_file()

checkpoint = torch.load(
    source_model,
    map_location="cpu",
    weights_only=True
)

assert len(checkpoint["classes"]) == 30
assert checkpoint["input_shape"] == [32, 150]

shutil.copy2(
    source_module,
    ml_dir / "karsl_inference.py"
)

shutil.copy2(
    source_model,
    models_dir / source_model.name
)

(delivery / "karsl" / "__init__.py").touch()
(ml_dir / "__init__.py").touch()

report_names = [
    "cnn_30_classes_v2_history.csv",
    "cnn_30_classes_v2_comparison.csv",
    "cnn_30_classes_v2_per_class_report.csv",
    "cnn_30_classes_v2_predictions.csv",
    "30_classes_model_comparison.csv"
]

for name in report_names:
    source = base / name
    assert source.is_file(), f"Missing report: {name}"
    shutil.copy2(source, reports_dir / name)

metadata = {
    "project": "SIGNOVA",
    "dataset": "KArSL-502 Pose",
    "model": "1D CNN",
    "model_version": "30_classes_v2",
    "status": "research_prototype",
    "classes": [int(c) for c in checkpoint["classes"]],
    "training_samples": 900,
    "validation_samples": 450,
    "input_shape": [32, 150],
    "preprocessing_version": checkpoint["preprocessing_version"],
    "best_epoch": int(checkpoint["best_epoch"]),
    "validation_accuracy": 0.2533,
    "validation_macro_f1": float(checkpoint["validation_macro_f1"]),
    "evaluation_split": "fold_1_validation_signer_2",
    "held_out_test_evaluated": False,
    "supports_live_camera": False
}

(ml_dir / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

(delivery / "requirements.txt").write_text(
    "numpy\n"
    "torch\n"
    "pose-format\n",
    encoding="utf-8"
)

readme = """# SIGNOVA — KArSL Machine Learning Prototype

## Overview

This package contains a research prototype for recognizing 30 isolated KArSL sign classes from Pose-format landmark sequences.

It is not a complete 502-class model or a real-time sign language translator.

## Model

- Architecture: 1D CNN
- Classes: Sign IDs 1–30
- Training samples: 900
- Validation samples: 450
- Validation accuracy: 25.33%
- Validation macro F1: 0.2231
- Best epoch: 17
- Evaluation: Signer-independent Fold 1 validation
- Held-out test evaluation: Not performed

## Input

A compatible KArSL `.pose` file or an already preprocessed NumPy array with shape `(32, 150)`.

The preprocessing uses 50 selected body and hand landmarks, X/Y coordinates, visibility indicators, shoulder normalization, and 32 sampled frames.

## Installation

```bash
pip install -r requirements.txt
```

## Example

Run from the delivery directory:

```python
from karsl.ml.karsl_inference import KArSLPredictor

predictor = KArSLPredictor(
    "karsl/ml/models/karsl_1d_cnn_30_classes_v2.pt"
)

results = predictor.predict_pose_file(
    "path/to/sample.pose",
    top_k=3
)

print(results)
```

## Output

```python
[
    {"sign_id": 1, "score": 0.7408},
    {"sign_id": 2, "score": 0.1514}
]
```

The example scores are illustrative of one previously tested sample.

Scores are softmax outputs, not calibrated confidence probabilities.

## Testing

```bash
python -m unittest discover -s tests -v
```

## Limitations

- Recognizes only 30 isolated signs.
- Some classes have zero validation F1.
- Cross-signer generalization remains weak.
- Requires a compatible Pose-format input.
- Live webcam-to-Pose processing is not implemented.
- Arabic label mapping is not included in the inference output.
- No held-out test results are reported.
- Correspondence between third-party Pose data and original videos has not been independently verified.

## Files

- `karsl/ml/karsl_inference.py`: preprocessing and prediction interface
- `karsl/ml/models/`: saved model checkpoint
- `karsl/ml/reports/`: training and validation results
- `karsl/ml/model_metadata.json`: model metadata
- `tests/`: basic inference tests
"""

(delivery / "README.md").write_text(
    readme,
    encoding="utf-8"
)

test_code = '''
import unittest
import numpy as np
from pathlib import Path
from karsl.ml.karsl_inference import KArSLPredictor


class TestKArSLInference(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        root = Path(__file__).resolve().parents[1]
        cls.predictor = KArSLPredictor(
            root / "karsl/ml/models/karsl_1d_cnn_30_classes_v2.pt"
        )

    def test_model_loading(self):
        self.assertEqual(len(self.predictor.classes), 30)

    def test_prediction_output(self):
        features = np.zeros((32, 150), dtype=np.float32)
        result = self.predictor.predict_features(features, top_k=3)
        self.assertEqual(len(result), 3)
        self.assertTrue(all(
            item["sign_id"] in self.predictor.classes
            for item in result
        ))
        self.assertTrue(all(
            0 <= item["score"] <= 1
            for item in result
        ))

    def test_invalid_shape(self):
        with self.assertRaises(ValueError):
            self.predictor.predict_features(
                np.zeros((32, 149), dtype=np.float32)
            )

    def test_invalid_values(self):
        features = np.full((32, 150), np.nan, dtype=np.float32)
        with self.assertRaises(ValueError):
            self.predictor.predict_features(features)

    def test_invalid_top_k(self):
        with self.assertRaises(ValueError):
            self.predictor.predict_features(
                np.zeros((32, 150), dtype=np.float32),
                top_k=31
            )


if __name__ == "__main__":
    unittest.main()
'''

(tests_dir / "test_inference.py").write_text(
    test_code.strip() + "\n",
    encoding="utf-8"
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-v"
    ],
    cwd=str(delivery),
    capture_output=True,
    text=True
)

print("=== PACKAGE TEST RESULTS ===")
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Package tests failed"

archive = shutil.make_archive(
    str(base / "SIGNOVA_KArSL_30class_prototype"),
    "zip",
    root_dir=delivery
)

print("\n=== FINAL DELIVERY STATUS ===")
print("Inference module: READY")
print("Trained model: READY")
print("Evaluation reports:", len(report_names))
print("README: READY")
print("Unit tests: PASS")
print("Package:", delivery)
print("ZIP:", archive)
print("Existing files modified: NO")
print("GitHub modified: NO")


In [ ]:
import json
import importlib.util
import numpy as np
import pandas as pd
import kagglehub
from pathlib import Path
from pose_format import Pose
from sklearn.metrics import accuracy_score, f1_score, classification_report

base = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")
test_cache = base / "pose_30_classes_test_v1.npz"
report_path = base / "cnn_30_classes_v2_test_report.csv"
metrics_path = base / "cnn_30_classes_v2_test_metrics.json"

module_path = base / "package/karsl_inference.py"
model_path = base / "karsl_1d_cnn_30_classes_v2.pt"

spec = importlib.util.spec_from_file_location("karsl_inference_test", module_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

predictor = module.KArSLPredictor(model_path)

fold = pose_folds[pose_folds["fold"] == "fold_1"].copy()

test_rows = fold[
    (fold["split"] == "test") &
    (fold["sign_id"].isin(predictor.classes))
].groupby("sign_id", group_keys=False).sample(
    n=15,
    random_state=42
)

test_rows = test_rows.sort_values(
    ["sign_id", "pose_file"]
).reset_index(drop=True)

paths = test_rows["pose_file"].astype(str).to_numpy(dtype=str)
labels = test_rows["sign_id"].astype(int).to_numpy()

assert len(paths) == 450
assert len(set(paths)) == 450

with np.load(base / "pose_30_classes_features_v2.npz", allow_pickle=True) as saved:
    development_paths = set(saved["paths"].astype(str))

assert development_paths.isdisjoint(paths)

development = fold[fold["pose_file"].isin(development_paths)]
assert set(development["signer_id"]).isdisjoint(set(test_rows["signer_id"]))

if test_cache.exists():
    with np.load(test_cache, allow_pickle=True) as saved:
        assert np.array_equal(saved["paths"].astype(str), paths)
        assert np.array_equal(saved["labels"], labels)
        assert str(saved["preprocessing_version"].item()) == module.PREPROCESSING_VERSION

        X_test = saved["X"].astype(np.float32).copy()
        done = saved["done"].astype(bool).copy()
else:
    X_test = np.zeros((450, 32, 150), dtype=np.float32)
    done = np.zeros(450, dtype=bool)

def save_test_progress():
    temporary = test_cache.with_suffix(".tmp.npz")

    np.savez_compressed(
        temporary,
        X=X_test,
        done=done,
        paths=paths,
        labels=labels,
        preprocessing_version=np.asarray(module.PREPROCESSING_VERSION)
    )

    temporary.replace(test_cache)

print("=== KARSL HELD-OUT TEST PREPARATION ===")
print("Classes:", len(predictor.classes))
print("Test samples:", len(paths))
print("Already processed:", int(done.sum()))
print("Remaining:", int((~done).sum()))

try:
    for i, pose_file in enumerate(paths):
        if done[i]:
            continue

        file_path = Path(
            kagglehub.dataset_download(
                "tfmohamedyahia/kars502-pose-75",
                path="content/kars502-pose-75k/" + pose_file
            )
        )

        with file_path.open("rb") as f:
            pose = Pose.read(f.read())

        features = module.preprocess_pose(pose)

        assert features.shape == (32, 150)
        assert np.isfinite(features).all()

        X_test[i] = features
        done[i] = True

        if int(done.sum()) % 25 == 0:
            save_test_progress()
            print(f"Processed: {int(done.sum())}/450")

finally:
    save_test_progress()

assert done.all()

predictions = [
    predictor.predict_features(features, top_k=1)[0]["sign_id"]
    for features in X_test
]

accuracy = accuracy_score(labels, predictions)
macro_f1 = f1_score(
    labels,
    predictions,
    labels=predictor.classes,
    average="macro",
    zero_division=0
)

report = classification_report(
    labels,
    predictions,
    labels=predictor.classes,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(report_path)

metrics = {
    "model": "karsl_1d_cnn_30_classes_v2",
    "split": "fold_1_test",
    "classes": len(predictor.classes),
    "test_samples": len(labels),
    "test_accuracy": float(accuracy),
    "test_macro_f1": float(macro_f1),
    "model_updated": False
}

metrics_path.write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

print("\n=== FINAL HELD-OUT TEST RESULT ===")
print("Test Accuracy:", round(accuracy * 100, 2), "%")
print("Test Macro F1:", round(macro_f1, 4))
print("Samples:", len(labels))
print("Test report saved: YES")
print("Test metrics saved: YES")
print("Model retrained: NO")
print("GitHub modified: NO")
print("Status: PASS")


## 6. Colab continuation: existing artifact checks


In [18]:
from google.colab import drive
from pathlib import Path
import numpy as np

drive.mount("/content/drive")

ML_BASE = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_pilot")

required_files = {
    "CNN checkpoint": ML_BASE / "karsl_1d_cnn_30_classes_v2.pt",
    "Development features": ML_BASE / "pose_30_classes_features_v2.npz",
    "Test features": ML_BASE / "pose_30_classes_test_v1.npz",
    "Inference module": ML_BASE / "package/karsl_inference.py",
}

missing = []

for name, path in required_files.items():
    exists = path.is_file()
    print(f"{name}: {'FOUND' if exists else 'MISSING'}")
    if not exists:
        missing.append(str(path))

if missing:
    raise FileNotFoundError("\n".join(missing))

with np.load(required_files["Development features"], allow_pickle=False) as cache:
    features = cache["X"]
    completed = cache["done"]

    assert features.shape == (1350, 32, 150), "Unexpected feature shape"
    assert completed.all(), "Preprocessing is incomplete"
    assert np.isfinite(features).all(), "Features contain invalid values"

    print("Development feature shape:", features.shape)
    print("Completed samples:", int(completed.sum()))

print("Status: READY TO CONTINUE")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CNN checkpoint: FOUND
Development features: FOUND
Test features: FOUND
Inference module: FOUND
Development feature shape: (1350, 32, 150)
Completed samples: 1350
Status: READY TO CONTINUE


## 7. Original private bundle verification and import


In [19]:
from google.colab import files
from pathlib import Path
from io import BytesIO
import hashlib
import zipfile

WORK_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_continuation")

if WORK_DIR.exists():
    raise FileExistsError(f"Workspace already exists: {WORK_DIR}")

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one ZIP file.")

bundle_name, bundle_data = next(iter(uploaded.items()))

expected_hash = "a56f2b1d480f67f431f4f8dc7e837a269bc207c539cccb9e791ffb79beca4905"

if hashlib.sha256(bundle_data).hexdigest() != expected_hash:
    raise ValueError("Please select the provided private reproducibility bundle.")

with zipfile.ZipFile(BytesIO(bundle_data)) as archive:
    if archive.testzip() is not None:
        raise ValueError("The ZIP file is corrupted.")

    WORK_DIR.mkdir(parents=True)
    archive.extractall(WORK_DIR)

gru_path = WORK_DIR / "karsl/ml/artifacts/private/karsl_gru_30class_v1.pt"

assert gru_path.is_file(), "GRU checkpoint was not found."

print("Bundle verification: PASS")
print("GRU checkpoint: FOUND")
print("Workspace:", WORK_DIR)
print("Status: READY FOR MODEL LOADING")


Saving SIGNOVA_KArSL_ML_PRIVATE_REPRO_BUNDLE (1).zip to SIGNOVA_KArSL_ML_PRIVATE_REPRO_BUNDLE (1).zip
Bundle verification: PASS
GRU checkpoint: FOUND
Workspace: /content/drive/MyDrive/SIGNOVA/KArSL/ml_continuation
Status: READY FOR MODEL LOADING


## 8. Selected GRU validation reproduction


In [20]:
import sys
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

WORK_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_continuation")
ML_DIR = WORK_DIR / "karsl/ml"

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

from karsl.ml.inference.predict import KArSLPredictor
from karsl.ml.src.data import load_30_class_data

gru_predictor = KArSLPredictor(
    model_path=ML_DIR / "artifacts/private/karsl_gru_30class_v1.pt",
    label_map_path=ML_DIR / "configs/label_map_30classes.json",
    calibration_path=ML_DIR / "configs/calibration_gru_validation_v1.json"
)

ml_data = load_30_class_data(
    folds_path=WORK_DIR / "karsl/data/pose/karsl_pose_signer_folds.csv",
    train_path=ML_DIR / "artifacts/private/pose_30_classes_features_v2.npz"
)

assert gru_predictor.classes == ml_data["classes"], "Class order mismatch"

X_validation, y_validation = ml_data["validation"]

validation_logits = gru_predictor.predict_logits(X_validation)

assert np.isfinite(validation_logits).all(), "Invalid model outputs"

validation_predictions = validation_logits.argmax(axis=1)
top5_indices = np.argsort(-validation_logits, axis=1)[:, :5]

validation_accuracy = accuracy_score(
    y_validation, validation_predictions
)

validation_macro_f1 = f1_score(
    y_validation,
    validation_predictions,
    labels=np.arange(len(gru_predictor.classes)),
    average="macro",
    zero_division=0
)

validation_top5 = (
    top5_indices == y_validation[:, None]
).any(axis=1).mean()

print("Model:", gru_predictor.architecture)
print("Model loading: PASS")
print("Classes:", len(gru_predictor.classes))
print("Validation samples:", len(y_validation))
print(f"Validation Accuracy: {validation_accuracy:.2%}")
print(f"Validation Macro F1: {validation_macro_f1:.4f}")
print(f"Validation Top-5 Accuracy: {validation_top5:.2%}")
print("Status: VALIDATION CHECK COMPLETED")


Model: SignGRU
Model loading: PASS
Classes: 30
Validation samples: 450
Validation Accuracy: 32.00%
Validation Macro F1: 0.2755
Validation Top-5 Accuracy: 62.44%
Status: VALIDATION CHECK COMPLETED


## 9. Initial package tests

The raw-Pose test was initially skipped because its fixture was missing. Sections 10 and 13 record the subsequent successful checks.


In [21]:
import sys
import subprocess
import importlib.util
from pathlib import Path

WORK_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_continuation")
REPORT_DIR = WORK_DIR / "karsl/ml/reports/colab_verification"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if importlib.util.find_spec("pytest") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "pytest>=7.4,<10"],
        check=True
    )

test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "karsl/ml/tests",
        "-q",
        "-ra",
        "--color=no"
    ],
    cwd=WORK_DIR,
    capture_output=True,
    text=True
)

test_output = test_result.stdout + "\n" + test_result.stderr
report_path = REPORT_DIR / "test_results_colab.txt"
report_path.write_text(test_output, encoding="utf-8")

print(test_output)
print("Test report:", report_path)

if test_result.returncode != 0:
    raise RuntimeError("Tests failed. Review the output above.")

print("Status: PACKAGE TESTS PASSED")


..............s..                                                        [100%]
=========================== short test summary info ============================
SKIPPED [1] karsl/ml/tests/test_ml.py:123: Set SIGNOVA_REAL_POSE to real source Pose file
16 passed, 1 skipped in 9.96s


Test report: /content/drive/MyDrive/SIGNOVA/KArSL/ml_continuation/karsl/ml/reports/colab_verification/test_results_colab.txt
Status: PACKAGE TESTS PASSED


## 10. Real Pose preprocessing parity and GRU inference


In [22]:
import os
import sys
import json
import shutil
import subprocess
import importlib.util
from pathlib import Path
import numpy as np

dependencies = {
    "kagglehub": "kagglehub",
    "pose_format": "pose-format>=0.6,<1"
}

missing_packages = [
    package
    for module, package in dependencies.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *missing_packages],
        check=True
    )

import kagglehub
from karsl.ml.src.preprocessing import load_pose_file, preprocess_pose

relative_pose_path = str(ml_data["paths"]["validation"][0])
saved_features = ml_data["validation"][0][0]

fixture_dir = ML_DIR / "artifacts/private/pose_fixture"
fixture_dir.mkdir(parents=True, exist_ok=True)
pose_path = fixture_dir / Path(relative_pose_path).name

if not pose_path.is_file():
    downloaded_path = kagglehub.dataset_download(
        "tfmohamedyahia/kars502-pose-75/versions/1",
        path="content/kars502-pose-75k/" + relative_pose_path
    )
    shutil.copy2(downloaded_path, pose_path)

recreated_features = preprocess_pose(load_pose_file(pose_path))

np.testing.assert_allclose(
    recreated_features,
    saved_features,
    atol=1e-6,
    rtol=1e-6
)

prediction = gru_predictor.predict_pose_file(pose_path, top_k=3)

test_environment = os.environ.copy()
test_environment["SIGNOVA_REAL_POSE"] = str(pose_path)

pose_test = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "karsl/ml/tests/test_ml.py::test_optional_real_pose_inference",
        "-q",
        "-ra",
        "--color=no"
    ],
    cwd=WORK_DIR,
    env=test_environment,
    capture_output=True,
    text=True
)

test_output = pose_test.stdout + "\n" + pose_test.stderr
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "real_pose_test.txt").write_text(
    test_output, encoding="utf-8"
)

print(test_output)

if pose_test.returncode != 0:
    raise RuntimeError("The real Pose test failed.")

report = {
    "status": "passed",
    "model_version": gru_predictor.model_version,
    "source_dataset_version": 1,
    "validation_sample": relative_pose_path,
    "feature_shape": list(recreated_features.shape),
    "maximum_feature_difference": float(
        np.max(np.abs(recreated_features - saved_features))
    ),
    "prediction": prediction
}

(REPORT_DIR / "gru_real_pose_check.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Preprocessing compatibility: PASS")
print("GRU real Pose inference: PASS")
print(json.dumps(prediction, indent=2, ensure_ascii=False))
print("Status: REAL POSE CHECK COMPLETED")


100%|██████████| 116k/116k [00:00<00:00, 3.87MB/s]


.                                                                        [100%]
1 passed in 3.62s


Preprocessing compatibility: PASS
GRU real Pose inference: PASS
{
  "status": "uncertain",
  "uncertain": true,
  "class_id": null,
  "label": null,
  "confidence": 0.271068,
  "top_k": [
    {
      "class_id": 2,
      "label": "1",
      "score": 0.271068
    },
    {
      "class_id": 1,
      "label": "0",
      "score": 0.18665
    },
    {
      "class_id": 7,
      "label": "6",
      "score": 0.130623
    }
  ],
  "model_version": "karsl_gru_30class_v1",
  "calibration_version": "validation_temperature_threshold_v1"
}
Status: REAL POSE CHECK COMPLETED


## 11. Independent smoke training and checkpoint checks


In [23]:
import os
import sys
import json
import hashlib
import subprocess

SMOKE_DIR = ML_DIR / "experiments/colab_smoke_verification"
selected_model_path = ML_DIR / "artifacts/private/karsl_gru_30class_v1.pt"

selected_model_hash = hashlib.sha256(
    selected_model_path.read_bytes()
).hexdigest()

training_environment = os.environ.copy()
training_environment["CUDA_VISIBLE_DEVICES"] = ""
training_environment["OMP_NUM_THREADS"] = "2"
training_environment["MKL_NUM_THREADS"] = "2"

smoke_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "karsl.ml.training.train",
        "--config",
        str(ML_DIR / "configs/gru_30class_v1.json"),
        "--folds",
        str(WORK_DIR / "karsl/data/pose/karsl_pose_signer_folds.csv"),
        "--train-cache",
        str(ML_DIR / "artifacts/private/pose_30_classes_features_v2.npz"),
        "--output",
        str(SMOKE_DIR),
        "--smoke"
    ],
    cwd=WORK_DIR,
    env=training_environment,
    capture_output=True,
    text=True
)

training_output = smoke_result.stdout + "\n" + smoke_result.stderr

(REPORT_DIR / "smoke_training_colab.txt").write_text(
    training_output, encoding="utf-8"
)

print(training_output)

if smoke_result.returncode != 0:
    raise RuntimeError("Smoke training failed.")

smoke_predictor = KArSLPredictor(SMOKE_DIR / "inference_best.pt")
smoke_prediction = smoke_predictor.predict_features(
    ml_data["validation"][0][0],
    top_k=3
)

assert len(smoke_prediction["top_k"]) == 3
assert (SMOKE_DIR / "resume_best.pt").is_file()
assert hashlib.sha256(
    selected_model_path.read_bytes()
).hexdigest() == selected_model_hash

smoke_run = json.loads((SMOKE_DIR / "run.json").read_text())
assert smoke_run["smoke"] is True
assert smoke_run["test_evaluated"] is False

print("Training script: PASS")
print("New checkpoint loading and prediction: PASS")
print("Resume checkpoint: FOUND")
print("Selected GRU checkpoint: UNCHANGED")
print("Status: SMOKE TRAINING PASSED")


Epoch 01 loss=3.6652 train_acc=0.0167 val_acc=0.0667 val_f1=0.0098
=== TRAINING RESULT ===
{
  "experiment_id": "gru_30classes_v1_local_2026-10-24",
  "architecture": "SignGRU",
  "smoke": true,
  "best_epoch": 1,
  "best_validation_macro_f1": 0.00975609756097561,
  "validation_accuracy_at_best": 0.06666666666666667,
  "training_seconds": 0.8707782189999307,
  "epochs_run": 1,
  "train_samples": 60,
  "validation_samples": 90,
  "selection": "validation_macro_f1",
  "seed": 42,
  "python": "3.13.16",
  "torch": "2.11.0+cpu",
  "device": "cpu",
  "os": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "test_evaluated": false,
  "optimizer_resumption_checkpoint": "resume_best.pt"
}


Training script: PASS
New checkpoint loading and prediction: PASS
Resume checkpoint: FOUND
Selected GRU checkpoint: UNCHANGED
Status: SMOKE TRAINING PASSED


## 12. English delivery preparation and artifact integrity


In [24]:
import ast
import csv
import json
import shutil
import hashlib
from pathlib import Path

DELIVERY_DIR = WORK_DIR.parent / "ml_delivery_english"

shutil.copytree(
    WORK_DIR,
    DELIVERY_DIR,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", "*.pyc")
)

DELIVERY_ML = DELIVERY_DIR / "karsl/ml"

def clean_code(source):
    tree = ast.parse(source)

    for node in ast.walk(tree):
        if isinstance(
            node,
            (ast.Module, ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)
        ):
            if ast.get_docstring(node, clean=False) is not None:
                node.body.pop(0)

                if not node.body and not isinstance(node, ast.Module):
                    node.body.append(ast.Pass())

    ast.fix_missing_locations(tree)
    cleaned = ast.unparse(tree) + "\n"

    assert ast.dump(tree, include_attributes=False) == ast.dump(
        ast.parse(cleaned), include_attributes=False
    )

    compile(cleaned, "<cleaned>", "exec")
    return cleaned

python_files = list(DELIVERY_ML.rglob("*.py"))

for path in python_files:
    path.write_text(
        clean_code(path.read_text(encoding="utf-8")),
        encoding="utf-8"
    )

notebook_cells = 0

for path in DELIVERY_ML.rglob("*.ipynb"):
    notebook = json.loads(path.read_text(encoding="utf-8"))

    for cell in notebook["cells"]:
        source = "".join(cell.get("source", []))

        if cell["cell_type"] == "code" and source.strip():
            cell["source"] = clean_code(source).splitlines(keepends=True)
            notebook_cells += 1

    path.write_text(
        json.dumps(notebook, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

summary = """# SIGNOVA KArSL ML Delivery

## Scope
A reproducible research prototype covering 30 isolated KArSL sign classes.
It is not a 502-class model, a verified Egyptian Sign Language model,
or a production translator.

## Selected model
Bidirectional GRU, selected using validation Macro F1.
Training: 900 samples. Validation: 450 samples. Test: 450 samples.

| Metric | Validation | Previously recorded test |
|---|---:|---:|
| Accuracy | 32.00% | 24.22% |
| Macro F1 | 0.2755 | 0.1860 |
| Top-5 Accuracy | 62.44% | 59.11% |

## Colab verification
The saved GRU validation metrics were reproduced.
The package suite passed 16 tests; its optional raw-Pose test subsequently passed.
A real validation Pose sample matched the cached preprocessing output.
The selected GRU successfully processed that Pose file.
Independent smoke training, checkpoint loading, and prediction passed.
The selected GRU checkpoint was unchanged.

## Deliverables
Preprocessing, training, evaluation, inference, configurations, tests,
experiment notebook, reports, model card, and reproduction instructions.

## Limitations
Accuracy remains low. Unknown-sign detection is unvalidated.
Data provenance, release rights, linguistic review, and backend agreement remain pending.
Historical CNN test exposure limits project-level test independence.
Private weights, feature arrays, and raw Pose files must remain separate from public code.

## Final verification
This delivery copy requires its final test run after code and documentation cleanup.
"""

(DELIVERY_ML / "DELIVERY_SUMMARY.md").write_text(
    summary, encoding="utf-8"
)

arabic_summary = DELIVERY_ML / "DELIVERY_SUMMARY_AR.md"

if arabic_summary.exists():
    arabic_summary.unlink()

for path in DELIVERY_ML.rglob("*.md"):
    text = path.read_text(encoding="utf-8")
    text = text.replace("DELIVERY_SUMMARY_AR.md", "DELIVERY_SUMMARY.md")
    text = text.replace("For an Arabic summary", "For the delivery summary")
    path.write_text(text, encoding="utf-8")

checklist_path = DELIVERY_ML / "requirements_checklist.csv"

with checklist_path.open(encoding="utf-8-sig", newline="") as stream:
    reader = csv.DictReader(stream)
    fields = reader.fieldnames
    rows = list(reader)

for row in rows:
    if row["requirement"] == "Real binary .pose integration in current environment":
        row["status"] = "DONE"
        row["evidence"] = (
            "reports/colab_verification/real_pose_test.txt; "
            "reports/colab_verification/gru_real_pose_check.json"
        )
        row["note"] = (
            "One real validation Pose sample passed preprocessing parity "
            "and GRU inference in Colab; this does not establish camera compatibility."
        )

with checklist_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

for source in WORK_DIR.rglob("*"):
    if source.is_file() and source.suffix in {".pt", ".npz", ".pose"}:
        target = DELIVERY_DIR / source.relative_to(WORK_DIR)
        assert hashlib.sha256(source.read_bytes()).digest() == hashlib.sha256(
            target.read_bytes()
        ).digest(), f"Artifact mismatch: {source.name}"

print("Python files cleaned:", len(python_files))
print("Notebook code cells cleaned:", notebook_cells)
print("English delivery summary: READY")
print("Real Pose checklist entry: UPDATED")
print("Model and data integrity: PASS")
print("Delivery folder:", DELIVERY_DIR)
print("Status: READY FOR FINAL TEST")


Python files cleaned: 19
Notebook code cells cleaned: 8
English delivery summary: READY
Real Pose checklist entry: UPDATED
Model and data integrity: PASS
Delivery folder: /content/drive/MyDrive/SIGNOVA/KArSL/ml_delivery_english
Status: READY FOR FINAL TEST


## 13. Final tests and selected GRU validation


In [25]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import os
import subprocess
import sys
import xml.etree.ElementTree as ET

DELIVERY_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_delivery_english")
DELIVERY_ML = DELIVERY_DIR / "karsl/ml"
FINAL_REPORT_DIR = DELIVERY_ML / "reports/colab_verification"
FINAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

pose_record = json.loads(
    (FINAL_REPORT_DIR / "gru_real_pose_check.json").read_text(encoding="utf-8")
)
real_pose = (
    DELIVERY_ML
    / "artifacts/private/pose_fixture"
    / Path(pose_record["validation_sample"]).name
)
assert real_pose.is_file(), f"Missing Pose fixture: {real_pose}"

env = os.environ.copy()
env.update({
    "PYTHONPATH": str(DELIVERY_DIR),
    "CUDA_VISIBLE_DEVICES": "",
    "OMP_NUM_THREADS": "2",
    "MKL_NUM_THREADS": "2",
    "SIGNOVA_REAL_POSE": str(real_pose),
    "SIGNOVA_ML_MODEL": str(
        DELIVERY_ML / "artifacts/private/karsl_1d_cnn_30_classes_v2.pt"
    ),
    "SIGNOVA_ML_TRAIN_CACHE": str(
        DELIVERY_ML / "artifacts/private/pose_30_classes_features_v2.npz"
    ),
    "SIGNOVA_ML_TEST_CACHE": str(
        DELIVERY_ML / "artifacts/private/pose_30_classes_test_v1.npz"
    ),
})

def run_check(command, log_path):
    result = subprocess.run(
        command,
        cwd=DELIVERY_DIR,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=300,
    )
    log_path.write_text(result.stdout, encoding="utf-8")
    if result.returncode != 0:
        print(result.stdout)
        raise RuntimeError(f"Verification failed. Log: {log_path}")
    return result.stdout

xml_path = FINAL_REPORT_DIR / "final_tests.xml"
test_output = run_check(
    [
        sys.executable, "-m", "pytest",
        "karsl/ml/tests", "-q", "-ra", "--color=no",
        f"--junitxml={xml_path}",
    ],
    FINAL_REPORT_DIR / "final_tests_colab.txt",
)
print(test_output.strip())

test_cases = list(ET.parse(xml_path).getroot().iter("testcase"))
skipped = sum(case.find("skipped") is not None for case in test_cases)
failed = sum(
    case.find("failure") is not None or case.find("error") is not None
    for case in test_cases
)
assert len(test_cases) >= 17 and skipped == 0 and failed == 0, (
    "The final suite must complete with all expected tests passing."
)

validation_dir = FINAL_REPORT_DIR / "gru_validation_final"
run_check(
    [
        sys.executable, "-m", "karsl.ml.evaluation.evaluate",
        "--model", str(
            DELIVERY_ML / "artifacts/private/karsl_gru_30class_v1.pt"
        ),
        "--folds", str(
            DELIVERY_DIR / "karsl/data/pose/karsl_pose_signer_folds.csv"
        ),
        "--train-cache", str(
            DELIVERY_ML / "artifacts/private/pose_30_classes_features_v2.npz"
        ),
        "--label-map", str(
            DELIVERY_ML / "configs/label_map_30classes.json"
        ),
        "--calibration", str(
            DELIVERY_ML / "configs/calibration_gru_validation_v1.json"
        ),
        "--split", "validation",
        "--output", str(validation_dir),
    ],
    FINAL_REPORT_DIR / "gru_validation_final.txt",
)

metrics = json.loads(
    (validation_dir / "validation_metrics.json").read_text(encoding="utf-8")
)
expected = {
    "accuracy": 0.32,
    "macro_f1": 0.27548809713573913,
    "top5_accuracy": 0.6244444444444445,
}
assert metrics["samples"] == 450
for name, value in expected.items():
    assert math.isclose(metrics[name], value, rel_tol=0, abs_tol=1e-6), (
        f"Validation mismatch: {name} = {metrics[name]}"
    )

verification = {
    "status": "passed",
    "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    "tests_passed": len(test_cases),
    "tests_skipped": skipped,
    "tests_failed": failed,
    "validation_samples": metrics["samples"],
    "validation_metrics": {name: metrics[name] for name in expected},
    "test_performance_evaluated": False,
    "training_performed": False,
}
(FINAL_REPORT_DIR / "final_verification.json").write_text(
    json.dumps(verification, indent=2),
    encoding="utf-8",
)

summary_path = DELIVERY_ML / "DELIVERY_SUMMARY.md"
summary = summary_path.read_text(encoding="utf-8")
summary = summary.partition("## Final verification")[0].rstrip()
summary += (
    "\n\n## Final verification\n"
    f"The cleaned delivery copy passed {len(test_cases)} tests with no skips.\n"
    "The suite included the real Pose inference check.\n"
    "The selected GRU reproduced its recorded validation metrics in a fresh process:\n"
    f"- Accuracy: {metrics['accuracy']:.2%}\n"
    f"- Macro F1: {metrics['macro_f1']:.4f}\n"
    f"- Top-5 Accuracy: {metrics['top5_accuracy']:.2%}\n"
    "No training or test-set performance evaluation was performed in this step.\n"
    "Evidence: reports/colab_verification/final_verification.json\n"
)
summary_path.write_text(summary, encoding="utf-8")

print(f"\nFinal tests: {len(test_cases)} passed, {skipped} skipped")
print(f"GRU Validation Accuracy: {metrics['accuracy']:.2%}")
print(f"GRU Validation Macro F1: {metrics['macro_f1']:.4f}")
print(f"GRU Validation Top-5 Accuracy: {metrics['top5_accuracy']:.2%}")
print("English delivery summary: UPDATED")
print("Status: FINAL VERIFICATION PASSED")


.................                                                        [100%]
- generated xml file: /content/drive/MyDrive/SIGNOVA/KArSL/ml_delivery_english/karsl/ml/reports/colab_verification/final_tests.xml -
17 passed in 11.08s

Final tests: 17 passed, 0 skipped
GRU Validation Accuracy: 32.00%
GRU Validation Macro F1: 0.2755
GRU Validation Top-5 Accuracy: 62.44%
English delivery summary: UPDATED
Status: FINAL VERIFICATION PASSED


## 14. Delivery archive creation

These recorded archive counts and sizes describe the packages before this notebook was added.


In [26]:
from pathlib import Path
import hashlib
import json
import zipfile

DELIVERY_DIR = Path("/content/drive/MyDrive/SIGNOVA/KArSL/ml_delivery_english")
DELIVERY_ML = DELIVERY_DIR / "karsl/ml"
PACKAGE_DIR = DELIVERY_DIR.parent / "ml_delivery_packages"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

verification_path = (
    DELIVERY_ML / "reports/colab_verification/final_verification.json"
)
verification = json.loads(verification_path.read_text(encoding="utf-8"))

assert verification["status"] == "passed"
assert verification["tests_passed"] >= 17
assert verification["tests_failed"] == 0
assert verification["tests_skipped"] == 0

notice = (
    "## Latest Colab delivery verification\n\n"
    f"The cleaned delivery passed {verification['tests_passed']} tests "
    "with no failures or skips, including real Pose inference.\n"
    "The selected GRU reproduced its recorded validation metrics.\n"
    "See [DELIVERY_SUMMARY.md](DELIVERY_SUMMARY.md) and "
    "[verification evidence]"
    "(reports/colab_verification/final_verification.json).\n\n"
    "Earlier environment descriptions below are historical records. "
    "The private archive now includes one validation Pose fixture; "
    "the code-only archive excludes raw Pose files, weights, and feature arrays.\n"
    "This remains a 30-class research prototype. "
    "502-class training, OOV validation, and backend agreement remain pending.\n\n"
)

for name in ["README.md", "PROJECT_STATE.md", "evaluation_report.md", "model_card.md"]:
    path = DELIVERY_ML / name
    content = path.read_text(encoding="utf-8")
    if "## Latest Colab delivery verification" not in content:
        heading, separator, body = content.partition("\n")
        path.write_text(
            heading + "\n\n" + notice + body.lstrip(),
            encoding="utf-8",
        )

excluded_dirs = {
    "__pycache__", ".pytest_cache", ".ipynb_checkpoints",
    ".git", ".venv", "venv",
}
public_extensions = {
    ".py", ".md", ".json", ".csv", ".txt", ".png", ".ipynb", ".xml",
}

def include_file(path):
    relative = path.relative_to(DELIVERY_DIR)
    return (
        path.is_file()
        and not any(part in excluded_dirs for part in relative.parts)
        and not any(
            part.startswith(".") and part != ".gitignore"
            for part in relative.parts
        )
        and path.suffix.lower() not in {".pyc", ".pyo"}
    )

private_files = sorted(
    path for path in DELIVERY_DIR.rglob("*") if include_file(path)
)
public_files = [
    path for path in private_files
    if path.is_relative_to(DELIVERY_ML)
    and "artifacts" not in path.relative_to(DELIVERY_ML).parts
    and (
        path.suffix.lower() in public_extensions
        or path.name == ".gitignore"
    )
]

required_public = {
    "karsl/ml/README.md",
    "karsl/ml/DELIVERY_SUMMARY.md",
    "karsl/ml/requirements.txt",
    "karsl/ml/requirements_checklist.csv",
    "karsl/ml/src/preprocessing.py",
    "karsl/ml/training/train.py",
    "karsl/ml/evaluation/evaluate.py",
    "karsl/ml/inference/predict.py",
    "karsl/ml/tests/test_ml.py",
    "karsl/ml/notebooks/06_KArSL_ML_Experiments.ipynb",
    "karsl/ml/reports/colab_verification/final_verification.json",
}
public_names = {
    path.relative_to(DELIVERY_DIR).as_posix() for path in public_files
}
assert required_public <= public_names, (
    f"Missing delivery files: {sorted(required_public - public_names)}"
)

def create_archive(filename, paths):
    target = PACKAGE_DIR / filename
    temporary = target.with_suffix(".zip.tmp")

    with zipfile.ZipFile(
        temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as archive:
        for path in paths:
            archive.write(path, path.relative_to(DELIVERY_DIR).as_posix())

    with zipfile.ZipFile(temporary) as archive:
        assert archive.testzip() is None, f"Archive verification failed: {filename}"
        assert len(archive.namelist()) == len(paths)

    temporary.replace(target)
    return target

code_archive = create_archive(
    "SIGNOVA_KArSL_ML_CODE_ONLY_EN.zip", public_files
)
private_archive = create_archive(
    "SIGNOVA_KArSL_ML_PRIVATE_REPRO_BUNDLE_EN.zip", private_files
)

checksum_lines = []
for path in [code_archive, private_archive]:
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    checksum_lines.append(f"{digest}  {path.name}")
    print(f"{path.name}: {path.stat().st_size / (1024 ** 2):.2f} MiB")

(PACKAGE_DIR / "SHA256SUMS.txt").write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)

print(f"\nCode-only files: {len(public_files)}")
print(f"Private bundle files: {len(private_files)}")
print("Archive integrity: PASS")
print("SHA256 checksums: SAVED")
print(f"Delivery location: {PACKAGE_DIR}")
print("Status: DELIVERY PACKAGES READY")


SIGNOVA_KArSL_ML_CODE_ONLY_EN.zip: 0.34 MiB
SIGNOVA_KArSL_ML_PRIVATE_REPRO_BUNDLE_EN.zip: 16.98 MiB

Code-only files: 108
Private bundle files: 125
Archive integrity: PASS
SHA256 checksums: SAVED
Delivery location: /content/drive/MyDrive/SIGNOVA/KArSL/ml_delivery_packages
Status: DELIVERY PACKAGES READY
